# GAN-RL Protein Function Prediction (Port-B-GAN)
**Goal:** Beat ProtHGT-ESM2 Biological Process Fmax (baseline: 0.7489)

**Where things stand:** Phase 1 (full-graph CompGCN pretraining) is complete and locked (`compgcn_pretrained_full_graph_LOCKED.pt` on Drive) -- you never need to retrain it. All active work is tuning the Phase 2 completion Generator on top of that frozen encoder.

**How this notebook is organized, top to bottom:**
- **Setup + Phase 1 pretraining** -- historical, already done, safe to skip.
- **"Root cause found" note** -- a real checkpoint-contamination bug from earlier in the project; worth reading if an old Fmax number won't reproduce.
- **Completion Generator baseline** -- the original hand-picked loss weights `{adv: 0.1, dm: 0.1, anchor: 1.0}`; kept as the "before" in every comparison.
- **"FULL RESET RECOVERY"** -- rebuilds everything Phase 2+ needs (no retraining) in one cell. **Run this first in any new/reconnected runtime.**
- **OFAT weight search -> final tuned pass -> Optuna search** -- the active work: finding better loss weights and training hyperparameters than the original guess, each searched more thoroughly than the last.
- **Optional utilities** (TensorBoard, resume-from-checkpoint) and the **deferred ProtHGT real-model comparison** near the end.

Cleaned up 2026-09-20: removed cells for approaches that were planned but never actually run (template Phase 2/3 cells, a calibration-head Phase 4 idea, shallow-KGE baseline cells) -- all superseded by the completion Generator work below. Nothing was deleted without a working alternative already in the notebook or recoverable from git history.

In [2]:
# ── Cell 1: Check GPU ──────────────────────────────────────────────────────────
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU')

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB


In [3]:
# ── Cell 2: Install Dependencies ───────────────────────────────────────────────
# Colab already has PyTorch — do NOT reinstall it.
# We detect the pre-installed torch/CUDA version and pull matching PyG wheels.
import subprocess, sys, torch

torch_ver = torch.__version__.split('+')[0]             # e.g. '2.3.0'
cuda_tag  = 'cu' + torch.version.cuda.replace('.', '') # e.g. 'cu121'
pyg_url   = f'https://data.pyg.org/whl/torch-{torch_ver}+{cuda_tag}.html'
print(f'Detected: torch={torch_ver}, cuda={cuda_tag}')
print(f'PyG wheel URL: {pyg_url}')

print('Installing torch_geometric ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch_geometric', '-q'], check=True)

print('Installing PyG sparse extensions ...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'torch_scatter', 'torch_sparse', 'torch_cluster',
    '-f', pyg_url, '-q'
], check=True)

print('Installing other deps ...')
subprocess.run([sys.executable, '-m', 'pip', 'install',
    'networkx', 'pyyaml', 'obonet',
    'scikit-learn', 'tqdm', 'pandas',
    'matplotlib', 'tensorboard', '-q'], check=True)

print('Done. No runtime restart needed.')

Detected: torch=2.11.0, cuda=cu128
PyG wheel URL: https://data.pyg.org/whl/torch-2.11.0+cu128.html
Installing torch_geometric ...
Installing PyG sparse extensions ...
Installing other deps ...
Done. No runtime restart needed.


In [4]:
# ── Cell 3: Mount Google Drive ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT     = '/content/drive/MyDrive/Poster code/prothgt/knowledge_graphs/'
CHECKPOINT_DIR = '/content/drive/MyDrive/Poster code/checkpoints/'
LOG_CSV        = os.path.join(CHECKPOINT_DIR, 'training_log.csv')

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Sanity check — verify ESM2 split files are in place
esm2_dir = os.path.join(DRIVE_ROOT, 'alternative_protein_embeddings/esm2/')
expected = [
    'prothgt-esm2-train-graph.pt',
    'prothgt-esm2-val-graph.pt',
    'prothgt-esm2-test-graph.pt',   # optional — val used as proxy if missing
]
for f in expected:
    path = os.path.join(esm2_dir, f)
    exists = os.path.exists(path)
    tag = 'OK' if exists else ('MISSING (optional)' if 'test' in f else 'MISSING')
    print(f'  {f}: {tag}')

print(f'\nDRIVE_ROOT:     {DRIVE_ROOT}')
print(f'CHECKPOINT_DIR: {CHECKPOINT_DIR}')

Mounted at /content/drive
  prothgt-esm2-train-graph.pt: OK
  prothgt-esm2-val-graph.pt: OK
  prothgt-esm2-test-graph.pt: OK

DRIVE_ROOT:     /content/drive/MyDrive/Poster code/prothgt/knowledge_graphs/
CHECKPOINT_DIR: /content/drive/MyDrive/Poster code/checkpoints/


In [5]:
# ── Cell 4: Clone Repository ───────────────────────────────────────────────────
import subprocess, sys, os

REPO_DIR = '/content/Prot_B_poster'
REPO_URL = 'https://github.com/Drjay806/Prot_B_poster.git'

if os.path.exists(REPO_DIR) and os.path.exists(os.path.join(REPO_DIR, 'src')):
    print('Repo already present — pulling latest ...')
    result = subprocess.run(['git', '-C', REPO_DIR, 'pull'], capture_output=True, text=True)
    print(result.stdout or result.stderr)
else:
    print('Cloning repo ...')
    result = subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], capture_output=True, text=True)
    if result.returncode != 0:
        print('ERROR cloning repo:')
        print(result.stderr)
        raise RuntimeError('Git clone failed — push your code first.')
    print(result.stdout)

# Install src as an editable package — fixes "No module named src" permanently.
# pip install -e registers src/ in Python's site-packages so any cell can import it
# without sys.path hacks, even after Drive remounts or kernel restarts within the session.
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-e', REPO_DIR, '-q'],
    capture_output=True, text=True
)
if result.returncode != 0:
    print('pip install -e failed:', result.stderr)
else:
    print('src package installed.')

print(f'Repo ready at {REPO_DIR}')
print('Contents:', os.listdir(REPO_DIR))

Cloning repo ...

src package installed.
Repo ready at /content/Prot_B_poster
Contents: ['src', 'configs', 'scripts', 'prot_b_poster.egg-info', 'requirements.txt', '.git', 'setup.py', 'notebooks']


In [6]:
# ── Cell 5: Load Config ────────────────────────────────────────────────────────
import yaml, os

config_path = os.path.join(REPO_DIR, 'configs/default.yaml')
with open(config_path) as f:
    cfg = yaml.safe_load(f)

# Override paths with what we set above
cfg['data']['drive_root'] = DRIVE_ROOT
cfg['data']['checkpoint_dir'] = CHECKPOINT_DIR

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
print('Target ontology:', cfg['data']['ontology'])
print('Target node type:', cfg['data']['target_type'])

Device: cuda
Target ontology: bp
Target node type: GO_term_P


In [17]:
import sys
sys.path.insert(0, REPO_DIR)

# ── Cell 6: Load Data + Build Hierarchy Tables ────────────────────────────────
# Data is loaded onto CPU. CompGCN moves only the relevant ~150 MB of node
# features to GPU per forward pass, freeing ~3 GB of VRAM for activations.

from src.data.loader import load_prothgt_splits
from src.data.go_hierarchy import build_ancestor_table, build_propagation_edges

splits = load_prothgt_splits(
    drive_root=cfg['data']['drive_root'],
    ontology=cfg['data']['ontology'],
    device='cpu',   # keep on CPU — encoder handles GPU placement internally
)
train_data, val_data, test_data = splits.train, splits.val, splits.test
target_type = splits.target_type

print(f'\nNode types: {train_data.node_types}')
print(f'Proteins:   {train_data["Protein"].x.shape[0]:,}')
print(f'GO terms:   {train_data[target_type].x.shape[0]:,}')

print('\nBuilding GO ancestor table (used by RL reward) ...')
ancestor_table = build_ancestor_table(train_data, target_type=target_type, cache=True)

print('Building propagation edge list (used by evaluation) ...')
prop_edges = build_propagation_edges(train_data, target_type=target_type, cache=True)
print(f'Ready. {len(ancestor_table):,} GO terms, {len(prop_edges):,} hierarchy edges.')

Loading train split from /content/drive/MyDrive/Poster code/prothgt/knowledge_graphs/alternative_protein_embeddings/esm2/prothgt-esm2-train-graph.pt ...
Loading val split from /content/drive/MyDrive/Poster code/prothgt/knowledge_graphs/alternative_protein_embeddings/esm2/prothgt-esm2-val-graph.pt ...
Loading test split from /content/drive/MyDrive/Poster code/prothgt/knowledge_graphs/alternative_protein_embeddings/esm2/prothgt-esm2-test-graph.pt ...
Loaded ProtHGT ESM2 splits  —  proteins: 261,373  GO terms (BP): 27,855

Node types: ['Protein', 'Disease', 'HPO', 'Drug', 'Compound', 'Domain', 'GO_term_P', 'GO_term_F', 'GO_term_C', 'Pathway', 'kegg_Pathway', 'EC_number']
Proteins:   261,373
GO terms:   27,855

Building GO ancestor table (used by RL reward) ...
Loading ancestor table from cache: /tmp/ancestor_table.pkl
Building propagation edge list (used by evaluation) ...
Ready. 27,855 GO terms, 64,409 hierarchy edges.


In [18]:
# Cell 6b: Verify Split Disjointness
# ProtHGT uses a transductive edge-level split: proteins are shared across splits
# (needed for GNN message passing), but the specific annotation edges are held out.
# We check that (protein, GO) pairs don't overlap between train and test.

# Clear .pyc cache so any freshly pulled source files are used
import subprocess, importlib
subprocess.run(["find", "/content/Prot_B_poster", "-name", "*.pyc", "-delete"], capture_output=True)

import src.data.graph_builder as _gb
importlib.reload(_gb)
from src.data.graph_builder import validate_split_disjointness

print("Checking split disjointness...")
for ont, ttype in [("bp", "GO_term_P"), ("mf", "GO_term_F"), ("cc", "GO_term_C")]:
    try:
        validate_split_disjointness(train_data, test_data, ttype)
    except (KeyError, ValueError) as e:
        print(f"  {ttype}: {e}")
print("Split validation complete.")


Checking split disjointness...
  GO_term_P: 16,327 proteins shared across splits (transductive edge-level split — expected for ProtHGT)
  GO_term_P: WARNING — 798 (0.8%) (protein, GO) pairs appear in both train and test supervision edges. This is a property of ProtHGT's split; both models see the same overlap so relative comparison remains valid.
  GO_term_P: 797,081 train edges / 101,384 test edges
  GO_term_F: 15,495 proteins shared across splits (transductive edge-level split — expected for ProtHGT)
  GO_term_F: WARNING — 4,179 (4.5%) (protein, GO) pairs appear in both train and test supervision edges. This is a property of ProtHGT's split; both models see the same overlap so relative comparison remains valid.
  GO_term_F: 741,034 train edges / 92,735 test edges
  GO_term_C: 13,186 proteins shared across splits (transductive edge-level split — expected for ProtHGT)
  GO_term_C: WARNING — 5,819 (8.6%) (protein, GO) pairs appear in both train and test supervision edges. This is a prop

In [19]:
# Cell 6c: Pre-compute Information Content Vectors (required for Smin metric)
# IC[t] = -log2(freq_train(t) / N_proteins), computed after label propagation.
# One call per ontology; results stored in ic_vecs dict keyed by ontology shortname.
from src.evaluation.smin import compute_information_content
from src.data.go_hierarchy import build_propagation_edges

ic_vecs = {}
for ont, ttype in [("bp", "GO_term_P"), ("mf", "GO_term_F"), ("cc", "GO_term_C")]:
    try:
        pe = build_propagation_edges(train_data, target_type=ttype, cache=True)
        ic = compute_information_content(train_data, ttype, pe)
        ic_vecs[ont] = ic
        print(f"  {ont} ({ttype}): IC ready ({(ic > 0).sum().item()} terms with annotations)")
    except Exception as e:
        print(f"  {ont}: skipped -- {e}")
print("IC computation complete.")


/content/Prot_B_poster/src/evaluation/smin.py:11: SyntaxWarning: invalid escape sequence '\ '
  ru(i,τ) = Σ_{g ∈ true_i \ pred_i(τ)}  IC(g)   — remaining uncertainty


  IC computed: 20,687/27,855 GO terms have annotations
  IC range: [0.62, 18.00]
  bp (GO_term_P): IC ready (20687 terms with annotations)
Built 13,295 propagation edges in topological order (GO_term_F)
  IC computed: 7,133/10,955 GO terms have annotations
  IC range: [0.98, 18.00]
  mf (GO_term_F): IC ready (7133 terms with annotations)
Built 6,127 propagation edges in topological order (GO_term_C)
  IC computed: 2,946/4,075 GO terms have annotations
  IC range: [0.68, 18.00]
  cc (GO_term_C): IC ready (2946 terms with annotations)
IC computation complete.


In [20]:
# ── Cell 7: Initialise Models ──────────────────────────────────────────────────
from src.models.compgcn import CompGCN
from src.models.generator import Generator
from src.models.discriminator import Discriminator
from src.models.distmult import DistMult
from src.models.reward import RewardModule
from src.utils.logger import TrainingLogger
from src.utils.seed import set_seed

set_seed(cfg.get('seed', 42))

encoder      = CompGCN(train_data, cfg).to(DEVICE)
generator    = Generator(cfg).to(DEVICE)
discriminator = Discriminator(cfg).to(DEVICE)
distmult     = DistMult(hidden_dim=cfg['distmult']['hidden_dim']).to(DEVICE)
reward_module = RewardModule(cfg, distmult, discriminator).to(DEVICE)

total_params = sum(p.numel() for p in encoder.parameters()) + \
               sum(p.numel() for p in generator.parameters()) + \
               sum(p.numel() for p in discriminator.parameters())
print(f'Total trainable parameters: {total_params:,}')

logger = TrainingLogger(
    log_dir='/tmp/runs',
    csv_path=LOG_CSV,
)
print('Logger ready. Run `%load_ext tensorboard` then `%tensorboard --logdir /tmp/runs` to monitor.')

Total trainable parameters: 2,355,201
Logger ready. Run `%load_ext tensorboard` then `%tensorboard --logdir /tmp/runs` to monitor.


## Full-Knowledge-Graph Retrain (all entities, no whitelist)

Everything below uses `configs/full_graph.yaml` instead of `default.yaml` — every node
type and edge type in the graph (Drug, Disease, Chembl, everything) is used for message
passing, with no relevance filtering. Requires a Premium GPU runtime (A100, 40GB) —
set this under Runtime > Change runtime type > GPU type before running these cells.

This builds a **separate** encoder (`encoder_full`) from a **separate** config
(`cfg_full`) so it never overwrites the whitelisted `encoder`/`cfg` from Cells 5-8
above — you can compare both runs side by side.
**Status: complete, historical.** This produced the locked checkpoint used by every Phase 2+ cell below; you do not need to re-run any of this unless you want to regenerate Phase 1 from scratch.

In [21]:
# ── Cell 8f: Load Full-Graph Config ──────────────────────────────────────────
import yaml, os

full_graph_config_path = os.path.join(REPO_DIR, 'configs/full_graph.yaml')
with open(full_graph_config_path) as f:
    cfg_full = yaml.safe_load(f)

cfg_full['data']['drive_root'] = DRIVE_ROOT
CHECKPOINT_DIR_FULL = cfg_full['data']['checkpoint_dir']
os.makedirs(CHECKPOINT_DIR_FULL, exist_ok=True)

whitelist = cfg_full['encoder']['gnn_edge_types']
print('gnn_edge_types:', whitelist if whitelist else '(empty — no filtering, all edges used)')
print('edge_chunk_size:', cfg_full['encoder']['edge_chunk_size'])
print('Checkpoint dir:', CHECKPOINT_DIR_FULL)

gnn_edge_types: (empty — no filtering, all edges used)
edge_chunk_size: 100000
Checkpoint dir: /content/drive/MyDrive/Poster code/checkpoints_full_graph/


In [22]:
# ── Cell 8g: Build CompGCN on the Full Graph ─────────────────────────────────
from src.models.compgcn import CompGCN
from src.utils.seed import set_seed

print('Relations present in train_data:', sorted(set(rel for _, rel, _ in train_data.edge_types)))

set_seed(cfg_full.get('seed', 42))

encoder_full = CompGCN(train_data, cfg_full).to(DEVICE)

n_full_params = sum(p.numel() for p in encoder_full.parameters())
print(f'Full-graph encoder parameters: {n_full_params:,}')
print(f'Node types projected (no whitelist): {list(encoder_full.input_projs.keys())}')

Relations present in train_data: ['Chembl', 'Disease', 'Drug', 'HPO', 'Orthology', 'PPI', 'Pathway', 'domain_function', 'function_function', 'hpodis', 'kegg_dis_drug', 'kegg_dis_path', 'kegg_dis_prot', 'kegg_path_prot', 'protein_domain', 'protein_ec', 'protein_function', 'rev_Chembl', 'rev_Disease', 'rev_Drug', 'rev_HPO', 'rev_Orthology', 'rev_PPI', 'rev_Pathway', 'rev_domain_function', 'rev_function_function', 'rev_hpodis', 'rev_kegg_dis_drug', 'rev_kegg_dis_path', 'rev_kegg_dis_prot', 'rev_kegg_path_prot', 'rev_protein_domain', 'rev_protein_ec', 'rev_protein_function']
Full-graph encoder parameters: 1,929,216
Node types projected (no whitelist): ['Protein', 'Disease', 'HPO', 'Drug', 'Compound', 'Domain', 'GO_term_P', 'GO_term_F', 'GO_term_C', 'Pathway', 'kegg_Pathway', 'EC_number']


In [23]:
# ── Cell 8h: Phase 1 on the Full Graph — Pre-training ──────────────────────────
# Same pretrain() function, same 4 losses as Cell 8 -- only the graph coverage and
# edge_chunk_size differ (see configs/full_graph.yaml).
#
# SAFE TO RE-RUN, ALWAYS: this cell builds a FRESH encoder_full from a seeded
# init every time it runs, instead of continuing to train whatever object the
# `encoder_full` variable currently holds. This matters because Phase 2
# (train_adversarial) updates its `encoder` argument's weights in place, and
# Cell 8k hands that object off as `encoder = encoder_full` -- a reference, not
# a copy. Without this guard, re-running this cell after any Phase 2 training
# would silently continue "pretraining" from an already-adversarially-updated
# encoder and overwrite the checkpoint with that blend. This is the confirmed
# root cause of several unreproducible Fmax numbers seen earlier in this
# project (0.4544 / 0.5415 / 0.5496 turning up where a clean ~0.3470 baseline
# was expected) -- see the markdown note below Cell 8k for the full writeup.
#
# Run a short epoch count first (edit cfg_full['pretrain']['epochs']) to confirm
# it fits in memory and val-Fmax isn't collapsed before committing to the full run.

from src.training.pretrain import pretrain
from src.utils.logger import TrainingLogger
from src.models.compgcn import CompGCN
from src.utils.seed import set_seed
import torch, os

set_seed(cfg_full.get('seed', 42))
encoder_full = CompGCN(train_data, cfg_full).to(DEVICE)

logger_full = TrainingLogger(
    log_dir='/tmp/runs_full_graph',
    csv_path=os.path.join(CHECKPOINT_DIR_FULL, 'training_log_full_graph.csv'),
)

encoder_full = pretrain(
    encoder=encoder_full,
    train_data=train_data,
    val_data=val_data,
    cfg=cfg_full,
    device=DEVICE,
    logger=logger_full,
)

# Primary checkpoint (plain filename downstream cells have always expected) ...
ckpt_path_full = os.path.join(CHECKPOINT_DIR_FULL, 'compgcn_pretrained_full_graph.pt')
torch.save(encoder_full.state_dict(), ckpt_path_full)

# ... plus a LOCKED backup that NO other cell in this notebook ever writes to.
# Every downstream reload (Phase 2 smoke test, Phase 2 full run) reads from this
# LOCKED file specifically, so even if the primary file above is ever
# accidentally contaminated again, Phase 2 can't be silently poisoned by it.
locked_path_full = os.path.join(CHECKPOINT_DIR_FULL, 'compgcn_pretrained_full_graph_LOCKED.pt')
torch.save(encoder_full.state_dict(), locked_path_full)

print(f'Saved full-graph pretrained encoder -> {ckpt_path_full}')
print(f'Saved LOCKED clean-baseline backup  -> {locked_path_full}')
print(f"Trained for {cfg_full['pretrain']['epochs']} epochs "
      f"(configs/full_graph.yaml pretrain.epochs) -- record this next to "
      f"any Fmax number measured on this checkpoint, since Phase 1's own "
      f"loss trains the same ComplEx score Fmax is computed from, so an "
      f"undertrained run and a fully-converged run are NOT comparable.")


Building annotation index ...
Building false-negative mask lookup ...
[rel_idx] available relations: ['Orthology', 'Pathway', 'kegg_path_prot', 'domain_function', 'function_function', 'protein_domain', 'PPI', 'HPO', 'kegg_dis_prot', 'Disease', 'Drug', 'kegg_dis_path', 'protein_ec', 'hpodis', 'kegg_dis_drug', 'Chembl', 'protein_function', 'rev_Orthology', 'rev_Pathway', 'rev_kegg_path_prot', 'rev_domain_function', 'rev_function_function', 'rev_protein_domain', 'rev_PPI', 'rev_HPO', 'rev_kegg_dis_prot', 'rev_Disease', 'rev_Drug', 'rev_kegg_dis_path', 'rev_protein_ec', 'rev_hpodis', 'rev_kegg_dis_drug', 'rev_Chembl', 'rev_protein_function']
[rel_idx] selected 'protein_function' → 16
Starting pre-training: 150 epochs, 797,081 positive pairs
  InfoNCE batch=8,192  temp=0.07  dm_neg=64
  Loss weights: 1.0*InfoNCE(sym+mask)  0.1*cosine  0.5*ComplEx-LP
  Node types the encoder projects (gnn_edge_types whitelist applied):
    Protein: 261,373 nodes × 1280d raw = 1338 MB
    Disease: 5,694 nodes

## Inspect the CompGCN Embeddings Directly

Nothing before this point ever exposes CompGCN's actual output vectors to you --
`pretrain()` and `evaluate_all()` both call `encoder(data)` internally, use the
result immediately, and discard it. This cell calls it once, standalone, and
holds onto the real embeddings so you can look at them directly, plus two
concrete checks that don't depend on the downstream Fmax task at all:

1. **Collapse check** -- are the embeddings degenerate (all nearly identical)?
2. **Graph-structure check** -- do proteins that are real PPI partners end up
   closer together in embedding space than random, unconnected pairs? If
   message passing is doing its job, the answer must be yes -- this has
   nothing to do with GO function prediction, it only tests whether the graph
   part of the Graph Neural Network is actually doing something.

In [24]:
# ── Cell 8h2: Inspect CompGCN Embeddings (3-hop, full-graph encoder) ─────────
import torch
import torch.nn.functional as F
import os

encoder_full.eval()
with torch.no_grad():
    protein_embs, go_embs, rel_embs = encoder_full(train_data)

print('=== Shapes ===')
print(f'Protein embeddings:  {tuple(protein_embs.shape)}')
print(f'GO term embeddings:  {tuple(go_embs.shape)}')
print(f'Relation embeddings: {tuple(rel_embs.shape)}')

p_norms = protein_embs.norm(dim=-1)
g_norms = go_embs.norm(dim=-1)
print('\n=== Collapse check ===')
print(f'Protein norm: mean={p_norms.mean():.4f}  std={p_norms.std():.4f}')
print(f'GO norm:      mean={g_norms.mean():.4f}  std={g_norms.std():.4f}')
if p_norms.mean() < 0.01 or p_norms.std() / (p_norms.mean() + 1e-8) < 0.01:
    print('WARNING: protein embeddings look collapsed (near-zero or near-identical norms).')
else:
    print('OK: protein embedding norms look healthy (non-zero, with real spread).')

print('\n=== Graph-structure check: are PPI partners closer than random pairs? ===')
ppi_key = ('Protein', 'PPI', 'Protein')
if ppi_key in train_data.edge_types:
    ppi_edges = train_data[ppi_key].edge_index
    n_sample  = min(5000, ppi_edges.shape[1])
    sel       = torch.randperm(ppi_edges.shape[1])[:n_sample]
    src, dst  = ppi_edges[0, sel], ppi_edges[1, sel]

    p_norm_vec = F.normalize(protein_embs.float(), dim=-1)
    connected_sim = (p_norm_vec[src] * p_norm_vec[dst]).sum(-1).mean().item()

    n_p = protein_embs.shape[0]
    rand_src = torch.randint(0, n_p, (n_sample,))
    rand_dst = torch.randint(0, n_p, (n_sample,))
    random_sim = (p_norm_vec[rand_src] * p_norm_vec[rand_dst]).sum(-1).mean().item()

    print(f'Connected (real PPI) pairs -- avg cosine similarity: {connected_sim:.4f}')
    print(f'Random, unconnected pairs  -- avg cosine similarity: {random_sim:.4f}')
    if connected_sim > random_sim:
        print('PASS: connected proteins are more similar than random -- graph structure is represented.')
    else:
        print('WARNING: connected proteins are NOT more similar than random -- message passing may not be working.')
else:
    print(f'No {ppi_key} edge type in train_data -- skipping this check.')

# Save the raw embeddings so they can be inspected outside this notebook too.
torch.save(protein_embs.cpu(), os.path.join(CHECKPOINT_DIR_FULL, 'protein_embeddings.pt'))
torch.save(go_embs.cpu(),      os.path.join(CHECKPOINT_DIR_FULL, 'go_embeddings.pt'))
print(f'\nSaved protein_embeddings.pt and go_embeddings.pt -> {CHECKPOINT_DIR_FULL}')

=== Shapes ===
Protein embeddings:  (261373, 256)
GO term embeddings:  (27855, 256)
Relation embeddings: (34, 256)

=== Collapse check ===
Protein norm: mean=16.0167  std=0.0328
GO norm:      mean=16.0281  std=0.0227

=== Graph-structure check: are PPI partners closer than random pairs? ===
Connected (real PPI) pairs -- avg cosine similarity: 0.3941
Random, unconnected pairs  -- avg cosine similarity: 0.1591
PASS: connected proteins are more similar than random -- graph structure is represented.

Saved protein_embeddings.pt and go_embeddings.pt -> /content/drive/MyDrive/Poster code/checkpoints_full_graph/


## Leakage-Aware Fmax Check

`validate_split_disjointness` (Cell 6b) already showed that some exact test
(protein, GO) triples also appear in train — 0.8% for BP, 4.5% for MF, 8.6% for CC.
That tells us the leak *rate*, not whether it actually inflates Fmax. This cell
answers that directly: score the same test set twice — once as-is, once with the
leaked triples removed — and compare. If Fmax barely moves, the overlap isn't doing
meaningful work and the baseline comparison stands; if it drops noticeably, that's a
real result to disclose rather than find out from a reviewer.

Set `EVAL_ENCODER` below to whichever encoder just finished training
(`encoder` for the whitelisted run, `encoder_full` for the full-graph run).

In [25]:
# ── Cell 8i: Leakage-Aware Fmax Check ────────────────────────────────────────
from src.data.graph_builder import find_leaked_pairs
from src.evaluation.metrics import evaluate_all

EVAL_ENCODER = encoder  # ← change to encoder_full to check the full-graph run instead

leaked_pairs = find_leaked_pairs(train_data, test_data, target_type)
print(f'Exact test triples also present in train: {len(leaked_pairs):,}')

print('\n--- Raw Fmax (as normally reported) ---')
raw_results = evaluate_all(
    encoder=EVAL_ENCODER, generator=None, distmult=distmult, data=test_data,
    ancestor_table=ancestor_table, target_type=target_type, cfg=cfg,
    device=DEVICE, mode='encoder', ic_vec=ic_vecs.get(cfg['data']['ontology']),
)

print('\n--- Leakage-free Fmax (leaked triples excluded from scoring) ---')
clean_results = evaluate_all(
    encoder=EVAL_ENCODER, generator=None, distmult=distmult, data=test_data,
    ancestor_table=ancestor_table, target_type=target_type, cfg=cfg,
    device=DEVICE, mode='encoder', ic_vec=ic_vecs.get(cfg['data']['ontology']),
    exclude_pairs=leaked_pairs,
)

delta = raw_results['fmax'] - clean_results['fmax']
print(f"\nRaw Fmax:          {raw_results['fmax']:.4f}")
print(f"Leakage-free Fmax: {clean_results['fmax']:.4f}")
print(f"Delta:             {delta:+.4f}", end=' ')
print('(small — overlap is not meaningfully inflating the score)' if abs(delta) < 0.005
      else '(non-trivial — disclose this explicitly when comparing to any baseline)')

Exact test triples also present in train: 798

--- Raw Fmax (as normally reported) ---
Encoding graph  [mode=encoder] ...
[rel_idx] available relations: ['Orthology', 'Pathway', 'kegg_path_prot', 'domain_function', 'function_function', 'protein_domain', 'PPI', 'HPO', 'kegg_dis_prot', 'Disease', 'Drug', 'kegg_dis_path', 'protein_ec', 'hpodis', 'kegg_dis_drug', 'Chembl', 'protein_function', 'rev_Orthology', 'rev_Pathway', 'rev_kegg_path_prot', 'rev_domain_function', 'rev_function_function', 'rev_protein_domain', 'rev_PPI', 'rev_HPO', 'rev_kegg_dis_prot', 'rev_Disease', 'rev_Drug', 'rev_kegg_dis_path', 'rev_protein_ec', 'rev_hpodis', 'rev_kegg_dis_drug', 'rev_Chembl', 'rev_protein_function']
[rel_idx] selected 'protein_function' → 16
Loading propagation edges ...
Computing Fmax+Smin [encoder] streaming 32,675 proteins ...
  Score range: [-1.667, 1.875]  (191 threshold candidates, extra resolution in top 10%)
Computing AUROC / AUPR / MCC (sample of 8,000 proteins) ...
Computing Hit@k / MRR

## Cheap Calibration Test — Full-Graph Phase 1 Encoder

The AUROC (0.887) vs. Fmax (0.165) gap above is the classic "ranks well, thresholds
poorly" pattern `train_calibration_head()` (Phase 4) exists to fix. It trains a small
MLP on the **frozen** `encoder_full` embeddings — cheap, no repeated graph forward
passes — then this cell re-scores Fmax via `mode='calibration'` so you can see how much
of that gap is recoverable before spending Phase 2/3 compute on the full graph.

Starts at 20 epochs for a quick look; raise to the default 50 if val-Fmax is still
climbing at the end of the printed log.

In [26]:
# ── Cell 8j: Quick Calibration Test on the Full-Graph Encoder ────────────────
from src.training.train_calibration import train_calibration_head
from src.evaluation.metrics import evaluate_all

calib_head_full = train_calibration_head(
    encoder=encoder_full,
    train_data=train_data,
    val_data=val_data,
    target_type=target_type,
    cfg=cfg_full,
    device=DEVICE,
    epochs=20,               # quick look first -- raise to 50 (default) if still improving
    checkpoint_dir=CHECKPOINT_DIR_FULL,
)

print('\n--- Fmax via calibration head (full-graph Phase 1 encoder) ---')
calib_results = evaluate_all(
    encoder=encoder_full, generator=None, distmult=distmult, data=test_data,
    ancestor_table=ancestor_table, target_type=target_type, cfg=cfg_full,
    device=DEVICE, mode='calibration', calibration_head=calib_head_full,
    ic_vec=ic_vecs.get(cfg_full['data']['ontology']),
)

print(f"\nComplEx (encoder-mode) Fmax: {raw_results['fmax']:.4f}   (from Cell 8i)")
print(f"Calibrated Fmax:             {calib_results['fmax']:.4f}")
print(f"Uplift from calibration:     {calib_results['fmax'] - raw_results['fmax']:+.4f}")

Caching encoder embeddings (frozen) ...
Training CalibrationHead: 20 epochs, 797,081 positives, neg_ratio=100, embed_dim=256
[CalibHead   1/20] loss=1.3896
[CalibHead   2/20] loss=1.0769
[CalibHead   3/20] loss=0.9968
[CalibHead   4/20] loss=0.9393
[CalibHead   5/20] loss=0.8953  val_Fmax=0.0121  *** NEW BEST
[CalibHead   6/20] loss=0.8582
[CalibHead   7/20] loss=0.8264
[CalibHead   8/20] loss=0.8020
[CalibHead   9/20] loss=0.7823
[CalibHead  10/20] loss=0.7622  val_Fmax=0.0090
[CalibHead  11/20] loss=0.7551
[CalibHead  12/20] loss=0.7313
[CalibHead  13/20] loss=0.7174
[CalibHead  14/20] loss=0.7086
[CalibHead  15/20] loss=0.6963  val_Fmax=0.0138  *** NEW BEST
[CalibHead  16/20] loss=0.6919
[CalibHead  17/20] loss=0.6837
[CalibHead  18/20] loss=0.6735
[CalibHead  19/20] loss=0.6629
[CalibHead  20/20] loss=0.6585  val_Fmax=0.0127

Done. Best val Fmax: 0.0138

--- Fmax via calibration head (full-graph Phase 1 encoder) ---
Encoding graph  [mode=calibration] ...
[rel_idx] available relatio

## Hand Off to the Full-Graph Encoder

Everything above this point built two separate Phase 1 encoders: the original
whitelisted `encoder` (Cell 8) and the enhanced full-graph `encoder_full`
(Cells 8f-8h, 3 hops, bigger training batches, seeded). Phase 2 onward is
written generically against the names `encoder` and `cfg` -- without this cell,
it would silently keep using the smaller whitelisted encoder and ignore
everything the full-graph run just did.

Run this cell to point the rest of the pipeline (Phase 2, 3, 4) at the
full-graph encoder. Skip it only if you deliberately want to continue with
the smaller whitelisted graph instead.

In [27]:
# ── Cell 8k: Hand Off to Full-Graph Encoder for Phase 2+ ────────────────────────
# Downstream cells (Phase 2, Phase 3, Cell 11d, Cell 12a/12b) all reference the
# PLAIN variables `encoder`, `cfg`, `CHECKPOINT_DIR`, `logger`, and the fixed
# filename 'compgcn_pretrained.pt' -- without reassigning ALL of these together:
#  - Phase 2/3 would save into the whitelisted checkpoint folder while actually
#    training the full-graph (3-layer) encoder, and Cell 11d/12a/12b would then
#    try to load the OLD 2-layer whitelisted checkpoint -- a shape-mismatch crash.
#  - Phase 2/3 would log into the SAME history dict Phase 1's whitelisted run
#    already logged 'val/fmax_bp' etc. into, tangling two different runs' points
#    together on the same step axis in Cell 9b/10b's plots.
#
# Note: `encoder = encoder_full` below is a REFERENCE, not a copy -- Phase 2/3
# cells further down don't rely on this alias, though. They each build their own
# fresh CompGCN and load it from the LOCKED checkpoint file directly (see Cell
# 8L and Cell 9), so nothing downstream can mutate `encoder_full` by accident.
import shutil, os

encoder        = encoder_full
cfg            = cfg_full
CHECKPOINT_DIR = CHECKPOINT_DIR_FULL
logger         = logger_full

# Downstream cells expect the Phase 1 checkpoint at the plain filename
# 'compgcn_pretrained.pt' inside CHECKPOINT_DIR -- copy it from the LOCKED
# backup (guaranteed clean, see Cell 8h) rather than the mutable primary file.
src_ckpt = os.path.join(CHECKPOINT_DIR_FULL, 'compgcn_pretrained_full_graph_LOCKED.pt')
dst_ckpt = os.path.join(CHECKPOINT_DIR_FULL, 'compgcn_pretrained.pt')
shutil.copyfile(src_ckpt, dst_ckpt)

print('Phase 2 onward will now train/evaluate/log on the full-graph, 3-hop encoder.')
print(f'Checkpoints will now save/load from: {CHECKPOINT_DIR}')
print(f"cfg['encoder']['num_layers'] = {cfg['encoder']['num_layers']}")

Phase 2 onward will now train/evaluate/log on the full-graph, 3-hop encoder.
Checkpoints will now save/load from: /content/drive/MyDrive/Poster code/checkpoints_full_graph/
cfg['encoder']['num_layers'] = 3


### Root cause found: the unreproducible Fmax numbers (0.4544 / 0.5415 / 0.5496)

Traced to a real bug in how Python objects alias, not to noise or a bad fix.
`encoder_full` was a single, mutable object built once in Cell 8g. Cell 8k
handed it off as `encoder = encoder_full` -- a *reference*, not a copy.
`train_adversarial()` (Phase 2) updates that object's weights in place. So
any time Cell 8h (Phase 1 pretrain) was re-run *after* Phase 2 had touched
`encoder_full` in the same Colab runtime, it would silently continue
"pretraining" from the already-adversarially-updated weights instead of a
fresh init -- then overwrite `compgcn_pretrained_full_graph.pt` with that
blend. Every later reload of "the clean Phase 1 checkpoint" was actually
loading whatever mixture of Phase 1 + partial Phase 2 happened to be on disk
at that moment, which is why the same file produced different, unreproducible
numbers across checks.

(DistMult was ruled out as a contamination source -- checked directly in
[src/models/distmult.py](../src/models/distmult.py): it has no learnable
parameters at all, so reusing that object across cells is harmless.)

**Fixed two ways, both in Cell 8h above:**
1. It now always constructs a brand-new `encoder_full` from a seeded init
   before training -- safe to re-run at any point, ever, with no risk of
   silently continuing from a contaminated state.
2. It now also saves a `_LOCKED` backup checkpoint that no other cell ever
   writes to. Every Phase 2 cell below (the smoke test and the full run)
   loads a fresh encoder from that LOCKED file specifically, so a
   contaminated primary file can never again silently poison Phase 2.
---

**Update, after re-running Cell 8h fresh in a new runtime:** the newly
fresh-trained LOCKED checkpoint scored Fmax ≈ 0.537 -- well above the
previously reported "clean baseline" of 0.3470. This is NOT new
contamination (the fresh-init guard above was in effect for this run).

Phase 1's own loss ([src/training/pretrain.py](../src/training/pretrain.py))
directly trains a ComplEx link-prediction ranking loss -- the exact scoring
function `evaluate_all`'s Fmax is computed from (`distmult_weight: 0.5` in
the loss mix). `configs/full_graph.yaml` even carries a note from earlier in
this project: an initial Phase 1 run was "a fit test, not a quality result"
-- i.e. a short run just to confirm the full graph fits in GPU memory, not a
real 150-epoch training run. The 0.3470 figure was very likely measured from
that short fit-test run, not a properly converged Phase 1 encoder.

**Implication:** the previously reported "Phase 2 gains" (0.3470 → 0.4544 →
0.5496) may be substantially overstated -- a real chunk of that apparent
improvement could simply be Phase 1 finally being allowed to finish, not
genuine adversarial lift. Going forward, judge Phase 2 (Cell 9) against
**this run's own Phase 1 baseline (~0.537, printed by Cell 8L below)**, not
the old 0.3470 figure -- if the full 100-epoch adversarial run doesn't clear
that bar by a meaningful margin, that's real evidence Phase 2 isn't adding
value beyond a properly-trained Phase 1 encoder, and is worth knowing before
writing up results.
---
**Note (2026-09-20 cleanup):** Cell 8L (the smoke test this discussion refers to) has been removed as superseded -- the real Phase 2 work now happens in the completion Generator cells below, all of which already load fresh from the LOCKED checkpoint per fix #2 above.

### Does the adversarial signal itself add anything? (encoder frozen this time)

The full 100-epoch run above showed the encoder-update rule causes a real,
sustained decline once it activates (confirmed: `adversarial_best.pt` was
saved at epoch 5 -- before the encoder update even starts -- and its REAL
(`evaluate_all`) Fmax is 0.5420, identical to the Phase 1 baseline; nothing
later in that run beat it). So the encoder-update rule doesn't belong in the
final architecture -- but that run never actually tested whether the
Generator/Critic adversarial game itself is useful, because every evaluation
so far has used `mode="encoder"`, which ignores the critic entirely.

This cell trains Phase 2 again with `freeze_encoder=True` -- the encoder
never changes from the Phase 1 checkpoint, so `mode="encoder"` is
guaranteed to reproduce 0.5420 exactly (a built-in sanity check). The real
question is whether `mode="critic"` (score pairs using the trained critic
directly) or `mode="ensemble"` (blend critic + ComplEx) beats that number.
If neither does, that's a clean, honest negative result for the adversarial
mechanism as a whole -- not just the encoder-update sub-piece.

In [28]:
import torch, os
from src.models.compgcn import CompGCN
from src.models.generator import Generator
from src.models.discriminator import Discriminator
from src.training.adversarial import train_adversarial
from src.evaluation.metrics import evaluate_all, tune_ensemble_alpha

locked_ckpt = os.path.join(CHECKPOINT_DIR, 'compgcn_pretrained_full_graph_LOCKED.pt')

frozen_encoder       = CompGCN(train_data, cfg).to(DEVICE)
frozen_encoder.load_state_dict(torch.load(locked_ckpt, map_location=DEVICE))
frozen_generator     = Generator(cfg).to(DEVICE)
frozen_discriminator = Discriminator(cfg).to(DEVICE)

frozen_cfg = dict(cfg)
frozen_cfg['adversarial'] = dict(cfg['adversarial'])
frozen_cfg['adversarial']['epochs'] = 100  # match the encoder-updating run for a fair comparison

frozen_encoder, frozen_generator, frozen_discriminator = train_adversarial(
    encoder=frozen_encoder, generator=frozen_generator, discriminator=frozen_discriminator,
    distmult=distmult, train_data=train_data, val_data=val_data,
    ancestor_table=ancestor_table, cfg=frozen_cfg, device=DEVICE,
    logger=logger, checkpoint_dir=None, freeze_encoder=True,
)

frozen_ckpt_path = os.path.join(CHECKPOINT_DIR, 'adversarial_frozen_encoder.pt')
torch.save({
    'encoder': frozen_encoder.state_dict(),
    'generator': frozen_generator.state_dict(),
    'discriminator': frozen_discriminator.state_dict(),
}, frozen_ckpt_path)
print(f'Saved frozen-encoder Phase 2 checkpoint -> {frozen_ckpt_path}')

# ── The real test: three scoring modes on the SAME checkpoint ──────────────
results = {}

results['encoder (sanity check, should equal 0.5420)'] = evaluate_all(
    encoder=frozen_encoder, generator=None, distmult=distmult, data=val_data,
    ancestor_table=ancestor_table, target_type=target_type, cfg=cfg,
    device=DEVICE, mode="encoder", ic_vec=ic_vecs.get(cfg["data"]["ontology"]),
)['fmax']

results['critic (does the trained critic alone rank better?)'] = evaluate_all(
    encoder=frozen_encoder, generator=None, distmult=distmult, data=val_data,
    ancestor_table=ancestor_table, target_type=target_type, cfg=cfg,
    device=DEVICE, mode="critic", discriminator=frozen_discriminator,
    ic_vec=ic_vecs.get(cfg["data"]["ontology"]),
)['fmax']

best_alpha = tune_ensemble_alpha(
    encoder=frozen_encoder, discriminator=frozen_discriminator, distmult=distmult,
    val_data=val_data, target_type=target_type, cfg=cfg, device=DEVICE,
)
print(f'Tuned ensemble alpha (ComplEx weight): {best_alpha:.2f}')
results[f'ensemble (alpha={best_alpha:.2f}, tuned on val)'] = evaluate_all(
    encoder=frozen_encoder, generator=None, distmult=distmult, data=val_data,
    ancestor_table=ancestor_table, target_type=target_type, cfg=cfg,
    device=DEVICE, mode="ensemble", discriminator=frozen_discriminator,
    ensemble_alpha=best_alpha, ic_vec=ic_vecs.get(cfg["data"]["ontology"]),
)['fmax']

print('\n' + '=' * 60)
print('  FROZEN-ENCODER PHASE 2 -- SCORING MODE COMPARISON')
print('=' * 60)
for name, fmax in results.items():
    print(f'  {name}: {fmax:.4f}')
print('=' * 60)
print('\nIf critic/ensemble > 0.5420, the adversarial mechanism adds real value.')
print('If not, that is an honest, isolated negative result for the mechanism as')
print('a whole -- not confounded with the encoder-update problem this time.')

[rel_idx] available relations: ['Orthology', 'Pathway', 'kegg_path_prot', 'domain_function', 'function_function', 'protein_domain', 'PPI', 'HPO', 'kegg_dis_prot', 'Disease', 'Drug', 'kegg_dis_path', 'protein_ec', 'hpodis', 'kegg_dis_drug', 'Chembl', 'protein_function', 'rev_Orthology', 'rev_Pathway', 'rev_kegg_path_prot', 'rev_domain_function', 'rev_function_function', 'rev_protein_domain', 'rev_PPI', 'rev_HPO', 'rev_kegg_dis_prot', 'rev_Disease', 'rev_Drug', 'rev_kegg_dis_path', 'rev_protein_ec', 'rev_hpodis', 'rev_kegg_dis_drug', 'rev_Chembl', 'rev_protein_function']
[rel_idx] selected 'protein_function' → 16
[NegativeSampler] tiers — easy: 9,481  medium: 9,403  hard: 8,971
Starting adversarial training: 100 epochs, 797,081 positive pairs
  WGAN + spectral norm | n_critic=5 | beta1=0.0 beta2=0.9
[Adv Epoch 1/100] W_dist=0.240  C_loss=-0.240  G_loss=-0.376  scores(real=0.09 fake=0.12 hard=-0.43)  DistMult=0.756  acc(real=75% rank=48%)  E_anchor=0.000  
[Adv Epoch 2/100] W_dist=0.249  

In [ ]:
# ── Optional: TensorBoard ──────────────────────────────────────────────────────
# Run this cell at any time to open TensorBoard and see live training curves.
%load_ext tensorboard
%tensorboard --logdir /tmp/runs/

In [ ]:
# ── Optional: Resume from Checkpoint ──────────────────────────────────────────
# If Colab disconnects mid-training, use this cell to reload from the last checkpoint.

RESUME_PHASE = 'adversarial'   # 'pretrain' | 'adversarial' | 'rl'
RESUME_PATH = os.path.join(CHECKPOINT_DIR, 'adversarial_checkpoint.pt')  # adjust as needed

ckpt = torch.load(RESUME_PATH, map_location=DEVICE)

if RESUME_PHASE == 'pretrain':
    encoder.load_state_dict(ckpt)
    print('Loaded pretrained encoder.')
elif RESUME_PHASE in ('adversarial', 'rl'):
    encoder.load_state_dict(ckpt['encoder'])
    generator.load_state_dict(ckpt['generator'])
    if 'discriminator' in ckpt:
        discriminator.load_state_dict(ckpt['discriminator'])
    print(f'Loaded {RESUME_PHASE} checkpoint.')

In [ ]:
# ── Optional: time of full graph run
import os, time
path = os.path.join(CHECKPOINT_DIR_FULL, 'compgcn_pretrained_full_graph.pt')
mtime = os.path.getmtime(path)
print("Last modified:", time.ctime(mtime))

## Optional: ProtHGT Real-Model Comparison (run separately, at the end)

Moved to the bottom deliberately -- these three cells CANNOT run as part of the
normal top-to-bottom pass above, and don't need to. Everything above this point
(Phases 1-4, all diagnostics, the poster table) is self-contained and complete
without this section.

**Why it's separate:** ProtHGT's released model checkpoint was built with
`torch_geometric==2.2.0`. The rest of this notebook needs a modern PyG version for
CompGCN. The two can't be installed in the same runtime at once -- loading
ProtHGT's checkpoint under a modern PyG fails with parameter-shape mismatches
(`HGTConv` was restructured internally between these versions). This isn't a bug
in our code; it's a real version conflict, confirmed by checking ProtHGT's own
`requirements.txt`.

**How to run this section, when you want ProtHGT's real score for comparison:**
1. Open a **separate, fresh Colab runtime** (Runtime > New runtime, or a throwaway
   notebook) -- not this one.
2. Run **Cell 11c-pre** there to confirm the correct ESM2 checkpoint file is on Drive.
3. Run **Cell 11c-inference** there (its header has the exact `pip install` command
   for the old, matching PyG version). This saves `prothgt_scores_bp.pt` to Drive.
4. Come back to **this** notebook/runtime and run **Cell 11c** below -- it only reads
   that saved score file, so it works fine here with the modern PyG install.

In [ ]:
# ── Cell 11c-pre: Run ProtHGT ESM2 BP Model and Save Scores ──────────────────
# Run this ONCE to generate prothgt_scores_bp.pt on Drive.
# After it completes, Cell 11c below feeds those scores through our grader.
#
# IMPORTANT: the model file must be the ESM2-specific checkpoint, not the default
# one -- ProtHGT ships several protein-embedding variants (APAAC/ESM2/TAPE/ProtT5)
# and only the ESM2 one (Protein input dim 1280) matches our own pipeline and the
# 0.7489 baseline number. On GitHub this lives at:
#   models/alternative_protein_embeddings/esm2/prothgt-esm2-model-biological-process.pt
# NOT models/prothgt-model-biological-process.pt (that one is 768-dim input --
# verified locally by loading both checkpoints and checking lin_dict.Protein.weight.shape).
import os, torch, sys

DRIVE_BASE = "/content/drive/MyDrive/Poster code/prothgt"
MODEL_PATH  = None

# Prefer a path containing 'esm2' explicitly -- fall back to any biological-process
# file only as a last resort, with a loud warning, since that's very likely the
# wrong (non-ESM2) variant.
candidates = []
for root, dirs, files in os.walk(DRIVE_BASE):
    for f in files:
        if "biological-process" in f and f.endswith(".pt"):
            candidates.append(os.path.join(root, f))

esm2_candidates = [c for c in candidates if "esm2" in c.lower()]
if esm2_candidates:
    MODEL_PATH = esm2_candidates[0]
elif candidates:
    MODEL_PATH = candidates[0]
    print(f"WARNING: no ESM2-specific checkpoint found under {DRIVE_BASE} -- "
          f"falling back to {MODEL_PATH}, which is very likely the wrong "
          f"(non-ESM2) protein-embedding variant. Download the real one from "
          f"models/alternative_protein_embeddings/esm2/ in the ProtHGT GitHub repo.")

if MODEL_PATH is None:
    raise FileNotFoundError(
        "No *-biological-process.pt found under "
        f"{DRIVE_BASE}. Download prothgt-esm2-model-biological-process.pt from "
        "https://github.com/HUBioDataLab/ProtHGT/tree/main/models/alternative_protein_embeddings/esm2 "
        "and place it under this Drive folder."
    )
print(f"Using model: {MODEL_PATH}")
print(f"  Size: {os.path.getsize(MODEL_PATH)/1e6:.1f} MB")

ckpt = torch.load(MODEL_PATH, map_location="cpu", weights_only=False)
protein_dim = ckpt['lin_dict.Protein.weight'].shape[1]
print(f"Protein input dim in this checkpoint: {protein_dim}")
assert protein_dim == 1280, (
    f"Expected 1280 (ESM2) but got {protein_dim} -- this is the wrong checkpoint variant. "
    f"Re-check MODEL_PATH above."
)
print("Confirmed: this is the ESM2 variant. Proceed to Cell 11c-inference --")
print("but read its header first: it must run in a SEPARATE Colab runtime, not this one.")

In [ ]:
# -- Cell 11c-inference: Run ProtHGT Inference (SEPARATE runtime required) --
#
# DO NOT run this in your main pipeline notebook/runtime. ProtHGT's checkpoint was
# saved under torch_geometric==2.2.0 (see their requirements.txt). Modern PyG
# (2.4+) restructured HGTConv internally -- separate per-node-type k_lin/q_lin/v_lin
# ModuleDicts became a single combined kqv_lin module. Loading this checkpoint under
# a modern PyG install fails with shape-mismatch errors on kqv_lin/k_lin/q_lin/v_lin --
# verified locally: strict-loading this exact checkpoint under PyG 2.6.1 produces
# hundreds of missing/unexpected keys in exactly that pattern. This is a real,
# version-level incompatibility, not a bug in the reconstruction below.
#
# Safe fix: run ONLY this cell in a fresh Colab runtime (Runtime > New runtime,
# or a throwaway notebook) with ProtHGT's exact original versions installed first:
#
#   !pip install -q torch==1.12.1+cpu torch_geometric==2.2.0 \
#       torch_scatter==2.1.0 torch_sparse==0.6.15 \
#       -f https://download.pytorch.org/whl/cpu/torch_stable.html \
#       -f https://data.pyg.org/whl/torch-1.12.0+cpu.html
#
# Then mount Drive, run this cell to generate and save prothgt_scores_bp.pt, and only
# THEN go back to your main (modern-PyG) runtime and run Cell 11c to grade those saved
# scores -- Cell 11c only loads a plain tensor from disk, so it never touches PyG
# version compatibility at all. This keeps the old/new PyG installs fully isolated.

import os, torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import HGTConv
from src.data.graph_builder import build_annotation_matrix

import torch_geometric
assert torch_geometric.__version__.startswith('2.2'), (
    f"torch_geometric=={torch_geometric.__version__} detected -- this cell requires "
    f"2.2.x to match how the checkpoint was saved. Run the pip install command in "
    f"this cell's header, in a FRESH runtime, before continuing."
)

MODEL_PATH = None
for root, dirs, files in os.walk('/content/drive/MyDrive/Poster code/prothgt'):
    for f in files:
        if 'esm2' in f.lower() and 'biological-process' in f and f.endswith('.pt'):
            MODEL_PATH = os.path.join(root, f); break
    if MODEL_PATH: break
print('Model:', MODEL_PATH)

SAVE_PATH = os.path.join(CHECKPOINT_DIR, 'prothgt_scores_bp.pt')
ckpt = torch.load(MODEL_PATH, map_location='cpu', weights_only=False)

# Extract exact metadata AND exact input dims from the checkpoint itself -- no guessing.
node_types = sorted(set(k.split('.')[1] for k in ckpt if k.startswith('lin_dict.')))
in_dims    = {nt: ckpt[f'lin_dict.{nt}.weight'].shape[1] for nt in node_types}
edge_type_strs = sorted(set(k[len('convs.0.a_rel.'):] for k in ckpt
                             if k.startswith('convs.0.a_rel.')))
edge_types = [tuple(s.split('__')) for s in edge_type_strs]
metadata   = (node_types, edge_types)
print('Node types:', len(node_types), '  Edge types:', len(edge_types))
print('Input dims:', in_dims)

class _MLP(nn.Module):
    def __init__(self):
        super().__init__()
        # matches mlp.lins.0/1/2/3 from state dict
        self.lins = nn.ModuleList([
            nn.Linear(256, 128), nn.Linear(128, 64),
            nn.Linear(64,  32),  nn.Linear(32,  1),
        ])
    def forward(self, x):
        for i, lin in enumerate(self.lins):
            x = lin(x)
            if i < len(self.lins) - 1:
                x = F.relu(x)
        return x

class ProtHGTInference(nn.Module):
    def __init__(self, in_dims, metadata, hidden=128, heads=8, layers=2):
        super().__init__()
        self.lin_dict = nn.ModuleDict({nt: nn.Linear(d, hidden) for nt, d in in_dims.items()})
        self.convs = nn.ModuleList([
            HGTConv(hidden, hidden, metadata, heads, group='sum')  # PyG 2.2.0 API
            for _ in range(layers)
        ])
        self.mlp = _MLP()

    def get_embeddings(self, data, device):
        x_dict = {}
        for nt, lin in self.lin_dict.items():
            if hasattr(data[nt], 'x') and data[nt].x is not None:
                x_dict[nt] = lin(data[nt].x.float().to(device)).relu()
        valid_et = set(edge_types)
        ei_dict  = {k: v.to(device) for k, v in data.edge_index_dict.items()
                    if k in valid_et}
        for conv in self.convs:
            x_dict = conv(x_dict, ei_dict)
        return x_dict

model = ProtHGTInference(in_dims, metadata)
missing, unexpected = model.load_state_dict(ckpt, strict=False)
print(f'Loaded. Missing: {len(missing)}  Unexpected: {len(unexpected)}')
if missing or unexpected:
    print('  Sample missing:', missing[:5])
    print('  Sample unexpected:', unexpected[:5])
    print('  If either list is non-empty here (with the correct 2.2.0 install), stop and')
    print('  investigate before trusting the scores -- this should be an exact match.')

model.to(DEVICE).eval()

print('Running ProtHGT forward pass (full graph)...')
with torch.no_grad():
    x_dict = model.get_embeddings(train_data, DEVICE)

protein_embs = x_dict['Protein'].cpu()
go_embs      = x_dict['GO_term_P'].cpu()
print('Protein embs:', protein_embs.shape)
print('GO_term_P embs:', go_embs.shape)

_, _, n_p, n_go = build_annotation_matrix(test_data, 'GO_term_P')
print(f'Scoring {n_p:,} proteins x {n_go:,} GO terms...')

scores = torch.zeros(n_p, n_go)
P, G   = 32, 1024   # protein chunk, GO chunk

with torch.no_grad():
    for i in range(0, n_p, P):
        end = min(i + P, n_p)
        C   = end - i
        p_chunk = protein_embs[i:end].to(DEVICE)
        for j in range(0, n_go, G):
            jend  = min(j + G, n_go)
            Gj    = jend - j
            g_chunk = go_embs[j:jend].to(DEVICE)
            p_exp = p_chunk.unsqueeze(1).expand(-1, Gj, -1).reshape(C * Gj, -1)
            g_exp = g_chunk.unsqueeze(0).expand(C, -1, -1).reshape(C * Gj, -1)
            probs = torch.sigmoid(
                model.mlp(torch.cat([p_exp, g_exp], dim=-1))
            ).squeeze(-1).reshape(C, Gj).cpu()
            scores[i:end, j:jend] = probs
        if i % (P * 100) == 0:
            print(f'  {i}/{n_p} proteins scored')

print(f'Score range: [{scores.min():.4f}, {scores.max():.4f}]')
torch.save(scores, SAVE_PATH)
print('Saved:', SAVE_PATH)
print('Now switch back to your MAIN (modern-PyG) runtime and run Cell 11c to grade these scores.')

In [ ]:
# -- Cell 11c: Evaluate ProtHGT Through Our Grader --
# Requires prothgt_scores_bp.pt saved by Cell 27 first.
# Self-contained: no dependency on all_splits or ablation_results.
import os, json, torch
from sklearn.metrics import roc_auc_score, matthews_corrcoef
from src.data.go_hierarchy import build_propagation_edges
from src.data.graph_builder import build_annotation_matrix
from src.evaluation.metrics import compute_ranking_metrics

SCORE_FILE = os.path.join(CHECKPOINT_DIR, 'prothgt_scores_bp.pt')
if not os.path.exists(SCORE_FILE):
    print('Score file not found:', SCORE_FILE)
    print('Run Cell 27 first.')
else:
    print('Loading', SCORE_FILE, '...')
    prothgt_scores = torch.load(SCORE_FILE).float()
    print('Shape:', prothgt_scores.shape)

    ttype = 'GO_term_P'
    row, col, n_p, n_go = build_annotation_matrix(test_data, ttype)
    true_mat = torch.zeros(n_p, n_go, dtype=torch.float32)
    true_mat[row.cpu(), col.cpu()] = 1.0

    prop_edges = build_propagation_edges(test_data, target_type=ttype, cache=True)
    for child_i, parent_i in prop_edges:
        prothgt_scores[:, parent_i] = torch.max(prothgt_scores[:, parent_i], prothgt_scores[:, child_i])
        true_mat[:, parent_i]       = torch.max(true_mat[:, parent_i],       true_mat[:, child_i])
    true_mat = (true_mat > 0.5).float()

    has_annot = true_mat.any(dim=1)
    sm = prothgt_scores[has_annot]; tm = true_mat[has_annot]
    ct_sum = tm.sum(1).clamp(min=1e-8)
    s_min_v, s_max_v = sm.min().item(), sm.max().item()
    t_steps = cfg.get('evaluation', {}).get('threshold_steps', 100)
    best_f1, best_t = 0.0, s_min_v
    for i in range(t_steps + 1):
        t = s_min_v + i * (s_max_v - s_min_v) / t_steps
        pred = (sm >= t).float()
        tp   = (pred * tm).sum(1)
        prec = (tp / pred.sum(1).clamp(1e-8)).mean().item()
        rec  = (tp / ct_sum).mean().item()
        if prec + rec > 0:
            f1 = 2 * prec * rec / (prec + rec)
            if f1 > best_f1: best_f1, best_t = f1, t

    unique_prots = row.cpu().unique()
    sel = unique_prots[torch.randperm(len(unique_prots))[:min(8000, len(unique_prots))]]
    auc_true   = true_mat[sel]; auc_scores = prothgt_scores[sel]
    col_has_pos  = auc_true.sum(0) > 0
    y_true_flat  = auc_true[:, col_has_pos].numpy().ravel()
    y_score_flat = auc_scores[:, col_has_pos].numpy().ravel()
    auroc = float(roc_auc_score(y_true_flat, y_score_flat)) if y_true_flat.sum() > 0 else 0.0
    rank_m = compute_ranking_metrics(auc_scores, auc_true)
    mcc = float(matthews_corrcoef(tm.numpy().astype(int).ravel(),
                                   (prothgt_scores[has_annot] >= best_t).numpy().astype(int).ravel()))

    res = {'fmax': best_f1, 'smin': -1, 'auroc': auroc, 'micro_f1': 0.0,
           'mcc': mcc, **rank_m}
    print('=== ProtHGT ESM2 (our grader, identical protocol) ===')
    print(f'  Fmax:   {best_f1:.4f}  (published under their eval: 0.7489)')
    print(f'  AUROC:  {auroc:.4f}')
    print(f'  Hit@10: {rank_m.get("hit@10", 0):.4f}')
    print(f'  MRR:    {rank_m.get("mrr", 0):.4f}')
    print(f'  MCC:    {mcc:.4f}')

    with open(os.path.join(CHECKPOINT_DIR, 'prothgt_grader_results.json'), 'w') as _f:
        json.dump({'bp': res}, _f, indent=2)
    print('Saved ->', os.path.join(CHECKPOINT_DIR, 'prothgt_grader_results.json'))


## Completion Generator -- baseline (original hand-picked weights)

Trains the Generator as an actual completion mechanism -- `{adv: 0.1, dm: 0.1, anchor: 1.0}` -- rather than just a hard-negative miner. This is the "before" in every later before/after comparison.

In [30]:
import torch, os, copy
from src.models.compgcn import CompGCN
from src.models.generator import Generator
from src.models.discriminator import Discriminator
from src.training.adversarial import train_adversarial
from src.evaluation.metrics import evaluate_all

locked_ckpt = os.path.join(CHECKPOINT_DIR, 'compgcn_pretrained_full_graph_LOCKED.pt')

gc_encoder       = CompGCN(train_data, cfg).to(DEVICE)
gc_encoder.load_state_dict(torch.load(locked_ckpt, map_location=DEVICE))
gc_generator     = Generator(cfg).to(DEVICE)
gc_discriminator = Discriminator(cfg).to(DEVICE)

gc_cfg = dict(cfg)
gc_cfg['adversarial'] = dict(cfg['adversarial'])
gc_cfg['adversarial']['epochs'] = 40  # shortened for time -- raise if you have room to spare

gc_encoder, gc_generator, gc_discriminator = train_adversarial(
    encoder=gc_encoder, generator=gc_generator, discriminator=gc_discriminator,
    distmult=distmult, train_data=train_data, val_data=val_data,
    ancestor_table=ancestor_table, cfg=gc_cfg, device=DEVICE,
    logger=None, checkpoint_dir=None, freeze_encoder=True,
    gen_loss_weights={'adv': 0.1, 'dm': 0.1, 'anchor': 1.0},
)

gc_ckpt_path = os.path.join(CHECKPOINT_DIR, 'generator_completion.pt')
torch.save({
    'encoder': gc_encoder.state_dict(),
    'generator': gc_generator.state_dict(),
    'discriminator': gc_discriminator.state_dict(),
}, gc_ckpt_path)
print(f'Saved generator-completion checkpoint -> {gc_ckpt_path}')

results = {}
results['encoder (sanity check, should equal 0.5462)'] = evaluate_all(
    encoder=gc_encoder, generator=None, distmult=distmult, data=val_data,
    ancestor_table=ancestor_table, target_type=target_type, cfg=cfg,
    device=DEVICE, mode="encoder", ic_vec=ic_vecs.get(cfg["data"]["ontology"]),
)['fmax']

print(f"BEFORE sweep -- encoder-only sanity check: {results['encoder (sanity check, should equal 0.5462)']:.4f}")

best_gw, best_fmax = 0.0, results['encoder (sanity check, should equal 0.5462)']
for gw in [0.3, 0.5, 0.7, 0.85, 1.0]:
    sweep_cfg = copy.deepcopy(cfg)
    sweep_cfg['evaluation']['gen_weight'] = gw
    r = evaluate_all(
        encoder=gc_encoder, generator=gc_generator, distmult=distmult, data=val_data,
        ancestor_table=ancestor_table, target_type=target_type, cfg=sweep_cfg,
        device=DEVICE, mode="encoder", ic_vec=ic_vecs.get(cfg["data"]["ontology"]),
    )
    print(f"  gen_weight={gw:.2f}: Fmax={r['fmax']:.4f}")
    if r['fmax'] > best_fmax:
        best_gw, best_fmax = gw, r['fmax']

print(f"\nBest: gen_weight={best_gw:.2f} -> Fmax={best_fmax:.4f} (vs. encoder-only baseline)")

[rel_idx] available relations: ['Orthology', 'Pathway', 'kegg_path_prot', 'domain_function', 'function_function', 'protein_domain', 'PPI', 'HPO', 'kegg_dis_prot', 'Disease', 'Drug', 'kegg_dis_path', 'protein_ec', 'hpodis', 'kegg_dis_drug', 'Chembl', 'protein_function', 'rev_Orthology', 'rev_Pathway', 'rev_kegg_path_prot', 'rev_domain_function', 'rev_function_function', 'rev_protein_domain', 'rev_PPI', 'rev_HPO', 'rev_kegg_dis_prot', 'rev_Disease', 'rev_Drug', 'rev_kegg_dis_path', 'rev_protein_ec', 'rev_hpodis', 'rev_kegg_dis_drug', 'rev_Chembl', 'rev_protein_function']
[rel_idx] selected 'protein_function' → 16
[NegativeSampler] tiers — easy: 9,481  medium: 9,403  hard: 8,971
Starting adversarial training: 40 epochs, 797,081 positive pairs
  WGAN + spectral norm | n_critic=5 | beta1=0.0 beta2=0.9
[Adv 1/40] W_dist=0.281  C_loss=-0.281  G_loss=0.335  E_loss=0.000  anchor=0.311  E_anchor=0.000  scores(real=0.38 fake=0.36 hard=-0.16)  DistMult=0.524  acc(real=88% rank=50%)
[Adv 2/40] W_di

## Overnight run -- baseline confirmation (superseded, kept for comparison)

Test-set confirmation + no-adversarial ablation + REINFORCE, all still using the *original* baseline weights above. Superseded by the tuned versions further down, but these numbers are the "before" half of the before/after comparison -- keep them.

In [33]:
import torch, os, copy, time, traceback
from src.models.compgcn import CompGCN
from src.models.generator import Generator
from src.models.discriminator import Discriminator
from src.models.reward import RewardModule
from src.training.adversarial import train_adversarial
from src.training.rl_trainer import train_rl_reinforce
from src.evaluation.metrics import evaluate_all

overnight_log = {}
t_start = time.time()
locked_ckpt = os.path.join(CHECKPOINT_DIR, 'compgcn_pretrained_full_graph_LOCKED.pt')
gc_ckpt_path = os.path.join(CHECKPOINT_DIR, 'generator_completion.pt')

def elapsed():
    return f'{(time.time() - t_start) / 60:.1f} min'

# ── STEP 1: Test-set confirmation for the completion run (fast, most decisive) ──
try:
    print('=' * 70); print('STEP 1: TEST-SET CONFIRMATION (completion run)'); print('=' * 70)
    gc_ckpt = torch.load(gc_ckpt_path, map_location=DEVICE)
    test_encoder = CompGCN(train_data, cfg).to(DEVICE)
    test_encoder.load_state_dict(gc_ckpt['encoder'])
    test_generator = Generator(cfg).to(DEVICE)
    test_generator.load_state_dict(gc_ckpt['generator'])

    for gw in [0.0, 0.85, 1.0]:
        test_cfg = copy.deepcopy(cfg)
        test_cfg['evaluation']['gen_weight'] = gw
        r = evaluate_all(
            encoder=test_encoder, generator=test_generator, distmult=distmult, data=test_data,
            ancestor_table=ancestor_table, target_type=target_type, cfg=test_cfg,
            device=DEVICE, mode='encoder', ic_vec=ic_vecs.get(cfg['data']['ontology']),
        )
        overnight_log[f'TEST-SET completion gen_weight={gw:.2f}'] = r
        print(f"  [TEST] gen_weight={gw:.2f}: Fmax={r['fmax']:.4f}  Smin={r['smin']:.4f}  "
              f"MCC={r['mcc']:.4f}  MicroF1={r['micro_f1']:.4f}")
    print(f'Step 1 done at {elapsed()}')
except Exception:
    print('STEP 1 FAILED:')
    traceback.print_exc()
print()

# ── STEP 2: No-adversarial ablation (train 40 epochs + val sweep) ──
try:
    print('=' * 70); print('STEP 2: NO-ADVERSARIAL ABLATION'); print('=' * 70)
    noadv_encoder = CompGCN(train_data, cfg).to(DEVICE)
    noadv_encoder.load_state_dict(torch.load(locked_ckpt, map_location=DEVICE))
    noadv_generator = Generator(cfg).to(DEVICE)
    noadv_discriminator = Discriminator(cfg).to(DEVICE)

    noadv_cfg = dict(cfg)
    noadv_cfg['adversarial'] = dict(cfg['adversarial'])
    noadv_cfg['adversarial']['epochs'] = 40

    noadv_encoder, noadv_generator, noadv_discriminator = train_adversarial(
        encoder=noadv_encoder, generator=noadv_generator, discriminator=noadv_discriminator,
        distmult=distmult, train_data=train_data, val_data=val_data,
        ancestor_table=ancestor_table, cfg=noadv_cfg, device=DEVICE,
        logger=None, checkpoint_dir=None, freeze_encoder=True,
        gen_loss_weights={'adv': 0.0, 'dm': 0.1, 'anchor': 1.0},
    )

    noadv_ckpt_path = os.path.join(CHECKPOINT_DIR, 'generator_completion_noadv.pt')
    torch.save({
        'encoder': noadv_encoder.state_dict(), 'generator': noadv_generator.state_dict(),
        'discriminator': noadv_discriminator.state_dict(),
    }, noadv_ckpt_path)
    print(f'Saved -> {noadv_ckpt_path}')

    for gw in [0.0, 0.3, 0.5, 0.7, 0.85, 1.0]:
        sweep_cfg = copy.deepcopy(cfg)
        sweep_cfg['evaluation']['gen_weight'] = gw
        r = evaluate_all(
            encoder=noadv_encoder, generator=noadv_generator, distmult=distmult, data=val_data,
            ancestor_table=ancestor_table, target_type=target_type, cfg=sweep_cfg,
            device=DEVICE, mode='encoder', ic_vec=ic_vecs.get(cfg['data']['ontology']),
        )
        overnight_log[f'VAL no-adv gen_weight={gw:.2f}'] = r
        print(f"  [VAL, no-adv] gen_weight={gw:.2f}: Fmax={r['fmax']:.4f}")
    print(f'Step 2 done at {elapsed()}')
except Exception:
    print('STEP 2 FAILED:')
    traceback.print_exc()
print()

# ── STEP 3: REINFORCE Phase 3, warm-started from the completion Generator ──
try:
    print('=' * 70); print('STEP 3: REINFORCE PHASE 3'); print('=' * 70)
    gc_ckpt = torch.load(gc_ckpt_path, map_location=DEVICE)
    rl_encoder = CompGCN(train_data, cfg).to(DEVICE)
    rl_encoder.load_state_dict(torch.load(locked_ckpt, map_location=DEVICE))
    rl_generator = Generator(cfg).to(DEVICE)
    rl_generator.load_state_dict(gc_ckpt['generator'])
    rl_discriminator = Discriminator(cfg).to(DEVICE)
    rl_discriminator.load_state_dict(gc_ckpt['discriminator'])
    print('Warm-started RL Generator/Critic from generator_completion.pt')

    reward_module = RewardModule(cfg, distmult, rl_discriminator).to(DEVICE)

    rl_cfg = dict(cfg)
    rl_cfg['rl'] = dict(cfg['rl'])
    rl_cfg['rl']['epochs'] = 30
    rl_cfg['rl']['policy_sigma'] = 0.1

    rl_encoder, rl_generator = train_rl_reinforce(
        encoder=rl_encoder, generator=rl_generator, distmult=distmult,
        reward_module=reward_module, train_data=train_data, val_data=val_data,
        ancestor_table=ancestor_table, cfg=rl_cfg, device=DEVICE,
        checkpoint_dir=CHECKPOINT_DIR, logger=None,
    )

    rl_ckpt_path = os.path.join(CHECKPOINT_DIR, 'rl_reinforce_final.pt')
    torch.save({'encoder': rl_encoder.state_dict(), 'generator': rl_generator.state_dict()}, rl_ckpt_path)
    print(f'Saved -> {rl_ckpt_path}')

    for gw in [0.0, 0.5, 0.85, 1.0]:
        sweep_cfg = copy.deepcopy(cfg)
        sweep_cfg['evaluation']['gen_weight'] = gw
        r = evaluate_all(
            encoder=rl_encoder, generator=rl_generator, distmult=distmult, data=val_data,
            ancestor_table=ancestor_table, target_type=target_type, cfg=sweep_cfg,
            device=DEVICE, mode='encoder', ic_vec=ic_vecs.get(cfg['data']['ontology']),
        )
        overnight_log[f'VAL reinforce gen_weight={gw:.2f}'] = r
        print(f"  [VAL, REINFORCE] gen_weight={gw:.2f}: Fmax={r['fmax']:.4f}")
    print(f'Step 3 done at {elapsed()}')
except Exception:
    print('STEP 3 FAILED:')
    traceback.print_exc()
print()

# ── FINAL SUMMARY ──
print('=' * 70); print('OVERNIGHT RUN COMPLETE -- SUMMARY'); print('=' * 70)
for k, v in overnight_log.items():
    print(f"  {k}: Fmax={v['fmax']:.4f}")
print(f'\nTotal time: {elapsed()}')

STEP 1: TEST-SET CONFIRMATION (completion run)
Encoding graph  [mode=encoder] ...
[rel_idx] available relations: ['Orthology', 'Pathway', 'kegg_path_prot', 'domain_function', 'function_function', 'protein_domain', 'PPI', 'HPO', 'kegg_dis_prot', 'Disease', 'Drug', 'kegg_dis_path', 'protein_ec', 'hpodis', 'kegg_dis_drug', 'Chembl', 'protein_function', 'rev_Orthology', 'rev_Pathway', 'rev_kegg_path_prot', 'rev_domain_function', 'rev_function_function', 'rev_protein_domain', 'rev_PPI', 'rev_HPO', 'rev_kegg_dis_prot', 'rev_Disease', 'rev_Drug', 'rev_kegg_dis_path', 'rev_protein_ec', 'rev_hpodis', 'rev_kegg_dis_drug', 'rev_Chembl', 'rev_protein_function']
[rel_idx] selected 'protein_function' → 16
Loading propagation edges ...
Computing Fmax+Smin [encoder] streaming 32,675 proteins ...
  Score range: [-6.548, 9.692]  (192 threshold candidates, extra resolution in top 10%)
Computing AUROC / AUPR / MCC (sample of 8,000 proteins) ...
Computing Hit@k / MRR (filtered ranking, KGC protocol) ...

 

## 🔁 START HERE after any runtime disconnect

Rebuilds everything Phase 2+ needs -- dependencies, Drive, repo, config, data splits, IC vectors, DistMult -- without retraining Phase 1. Run this once per new/reconnected runtime, then continue with whichever cell below you were working on.

In [7]:
# ============================================================
# FULL RESET RECOVERY -- rebuilds everything needed, no retraining
# ============================================================

# --- Install dependencies ---
import subprocess, sys, torch
torch_ver = torch.__version__.split('+')[0]
cuda_tag  = 'cu' + torch.version.cuda.replace('.', '')
pyg_url   = f'https://data.pyg.org/whl/torch-{torch_ver}+{cuda_tag}.html'
subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch_geometric', '-q'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch_scatter', 'torch_sparse', 'torch_cluster', '-f', pyg_url, '-q'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'networkx', 'pyyaml', 'obonet', 'scikit-learn', 'tqdm', 'pandas', 'matplotlib', 'tensorboard', '-q'], check=True)
print("Dependencies installed.")

# --- Mount Drive ---
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_ROOT = '/content/drive/MyDrive/Poster code/prothgt/knowledge_graphs/'

# --- Clone repo ---
REPO_DIR = '/content/Prot_B_poster'
REPO_URL = 'https://github.com/Drjay806/Prot_B_poster.git'
if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/main'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', REPO_DIR, '-q'], check=True)
print(subprocess.run(['git', '-C', REPO_DIR, 'log', '-1', '--oneline'], capture_output=True, text=True).stdout)

import sys
sys.path.insert(0, REPO_DIR)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- Load full-graph config directly ---
import yaml
with open(os.path.join(REPO_DIR, 'configs/full_graph.yaml')) as f:
    cfg = yaml.safe_load(f)
cfg['data']['drive_root'] = DRIVE_ROOT
CHECKPOINT_DIR = cfg['data']['checkpoint_dir']
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)

# --- Load data splits ---
from src.data.loader import load_prothgt_splits
from src.data.go_hierarchy import build_ancestor_table, build_propagation_edges

splits = load_prothgt_splits(drive_root=cfg['data']['drive_root'], ontology=cfg['data']['ontology'], device='cpu')
train_data, val_data, test_data = splits.train, splits.val, splits.test
target_type = splits.target_type
ancestor_table = build_ancestor_table(train_data, target_type=target_type, cache=True)

# --- IC vectors (needed for Smin) ---
from src.evaluation.smin import compute_information_content
ic_vecs = {}
for ont, ttype in [("bp", "GO_term_P"), ("mf", "GO_term_F"), ("cc", "GO_term_C")]:
    try:
        pe = build_propagation_edges(train_data, target_type=ttype, cache=True)
        ic_vecs[ont] = compute_information_content(train_data, ttype, pe)
    except Exception as e:
        print(f"  {ont}: skipped -- {e}")

# --- DistMult (stateless, no training needed) ---
from src.models.distmult import DistMult
distmult = DistMult(hidden_dim=cfg['distmult']['hidden_dim']).to(DEVICE)

print("\nSetup complete -- ready for the weight-search cell.")

Dependencies installed.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
bf432ce feat: add OFAT weight-sensitivity search (adv -> dm -> anchor) + final tuned run

CHECKPOINT_DIR: /content/drive/MyDrive/Poster code/checkpoints_full_graph/
Loading train split from /content/drive/MyDrive/Poster code/prothgt/knowledge_graphs/alternative_protein_embeddings/esm2/prothgt-esm2-train-graph.pt ...
Loading val split from /content/drive/MyDrive/Poster code/prothgt/knowledge_graphs/alternative_protein_embeddings/esm2/prothgt-esm2-val-graph.pt ...
Loading test split from /content/drive/MyDrive/Poster code/prothgt/knowledge_graphs/alternative_protein_embeddings/esm2/prothgt-esm2-test-graph.pt ...
Loaded ProtHGT ESM2 splits  —  proteins: 261,373  GO terms (BP): 27,855
Building ancestor table for 27,855 GO terms (GO_term_P) ...
Done. Avg ancestors per GO term: 21.4
Ancestor table cached to /tmp/ancestor_table.pkl


/content/Prot_B_poster/src/evaluation/smin.py:11: SyntaxWarning: invalid escape sequence '\ '
  ru(i,τ) = Σ_{g ∈ true_i \ pred_i(τ)}  IC(g)   — remaining uncertainty


Built 64,409 propagation edges in topological order (GO_term_P)
  IC computed: 20,687/27,855 GO terms have annotations
  IC range: [0.62, 18.00]
Built 13,295 propagation edges in topological order (GO_term_F)
  IC computed: 7,133/10,955 GO terms have annotations
  IC range: [0.98, 18.00]
Built 6,127 propagation edges in topological order (GO_term_C)
  IC computed: 2,946/4,075 GO terms have annotations
  IC range: [0.68, 18.00]

Setup complete -- ready for the weight-search cell.


### Corrected + extended leakage check (train vs. val AND train vs. test)

The original Phase-1-section leakage cell ("Cell 8i", up in the historical section above) has a confirmed bug: it evaluated `EVAL_ENCODER = encoder` *before* Cell 8k's hand-off ever ran, so it silently scored the untrained whitelisted-graph encoder instead of the real, 150-epoch full-graph one -- explaining its Fmax=0.1007 result, wildly below every other number in this project. `validate_split_disjointness` (used by Cell 6b) has a separate problem for this purpose: its print statements hardcode the word "test" no matter what you actually pass it, so reusing it to check val would print output that lies about what it's describing.

This cell is deliberately placed **here**, right after FULL RESET RECOVERY, instead of back in the old Phase-1 section -- that positioning is exactly what caused the original bug (plain `cfg`/`CHECKPOINT_DIR` meant something different depending on where in the notebook you were). Here, both are already and unambiguously the full-graph versions, the same as every other Phase 2+ cell in this notebook relies on.

Three safeguards against repeating the same mistake:
1. Loads the encoder **explicitly from the named LOCKED checkpoint file** -- no ambient `encoder`/`encoder_full` variable, so no aliasing bug is possible. Prints the checkpoint path and parameter count so it's verifiable by eye.
2. Hand-writes correctly-labeled prints for train-vs-val, train-vs-test, and val-vs-test overlap using `build_annotation_matrix` directly -- no borrowed hardcoded "test" strings.
3. **Cross-checks against a known reference:** every prior run in this project has shown this encoder's raw (no-generator) Fmax at approximately 0.5462 (val) / 0.5501 (test). An `assert` fails loudly if this cell's raw numbers don't land near those -- the same check that would have caught the original bug immediately (0.10 vs. an expected ~0.55).

Covers both remaining splits (val and test) on BP (GO_term_P), the only ontology this project actually uses.

In [ ]:
import torch, os
from src.models.compgcn import CompGCN
from src.data.graph_builder import build_annotation_matrix, find_leaked_pairs
from src.evaluation.metrics import evaluate_all

# -- Step 1: load the encoder EXPLICITLY from the named checkpoint -- no ambient variable --
locked_ckpt = os.path.join(CHECKPOINT_DIR, 'compgcn_pretrained_full_graph_LOCKED.pt')
assert os.path.exists(locked_ckpt), f'Checkpoint not found: {locked_ckpt}'
print(f'Loading encoder from: {locked_ckpt}')

verify_encoder = CompGCN(train_data, cfg).to(DEVICE)
verify_encoder.load_state_dict(torch.load(locked_ckpt, map_location=DEVICE))
verify_encoder.eval()
n_params = sum(p.numel() for p in verify_encoder.parameters())
print(f'Loaded OK -- {n_params:,} parameters. (Sanity: should be a few million, not near-zero or huge.)')

# -- Step 2: pairwise split overlap, BP only, with correctly labeled prints --
print('\n' + '=' * 70)
print('SPLIT OVERLAP -- GO_term_P (BP)')
print('=' * 70)

splits_to_check = {'train': train_data, 'val': val_data, 'test': test_data}
pairs_by_split = {}
for name, d in splits_to_check.items():
    row, col, n_p, n_go = build_annotation_matrix(d, 'GO_term_P')
    pairs_by_split[name] = set(zip(row.cpu().tolist(), col.cpu().tolist()))
    print(f'  {name}: {len(pairs_by_split[name]):,} positive (protein, GO) edges')

print()
for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
    overlap = pairs_by_split[a] & pairs_by_split[b]
    pct = 100 * len(overlap) / max(len(pairs_by_split[b]), 1)
    print(f'  {a} vs {b}: {len(overlap):,} overlapping (protein, GO) pairs ({pct:.2f}% of {b})')

# -- Step 3: leakage-aware Fmax, BOTH val and test, on the explicitly-loaded encoder --
EXPECTED_RAW_FMAX = {'val': 0.5462, 'test': 0.5501}   # from every prior run in this project

for split_name, split_data in [('val', val_data), ('test', test_data)]:
    print('\n' + '=' * 70)
    print(f'LEAKAGE-AWARE FMAX -- {split_name.upper()} (BP)')
    print('=' * 70)

    leaked = find_leaked_pairs(train_data, split_data, 'GO_term_P')
    print(f'{split_name}: {len(leaked):,} exact triples also present in train')

    raw = evaluate_all(
        encoder=verify_encoder, generator=None, distmult=distmult, data=split_data,
        ancestor_table=ancestor_table, target_type='GO_term_P', cfg=cfg,
        device=DEVICE, mode='encoder', ic_vec=ic_vecs.get('bp'),
    )
    clean = evaluate_all(
        encoder=verify_encoder, generator=None, distmult=distmult, data=split_data,
        ancestor_table=ancestor_table, target_type='GO_term_P', cfg=cfg,
        device=DEVICE, mode='encoder', ic_vec=ic_vecs.get('bp'),
        exclude_pairs=leaked,
    )

    expected = EXPECTED_RAW_FMAX[split_name]
    sanity_ok = abs(raw['fmax'] - expected) < 0.02
    status = 'OK, matches prior runs' if sanity_ok else 'MISMATCH -- wrong encoder loaded, stop and investigate'
    print(f"\nRaw Fmax:          {raw['fmax']:.4f}  (expected ~{expected:.4f} -- {status})")
    print(f"Leakage-free Fmax: {clean['fmax']:.4f}")
    delta = raw['fmax'] - clean['fmax']
    note = '(small -- overlap is not meaningfully inflating the score)' if abs(delta) < 0.005 else '(non-trivial -- disclose this explicitly)'
    print(f"Delta:             {delta:+.4f} {note}")
    fail_msg = f'{split_name}: raw Fmax {raw["fmax"]:.4f} does not match the expected ~{expected:.4f} -- wrong encoder or wrong data, stop here.'
    assert sanity_ok, fail_msg

### Test-split data overview (for the paper's dataset description)

Pure data reporting -- no model, no encoder, so none of the risk from the leakage-check bugs applies here. Answers, precisely, what's actually inside the test split you're predicting on:

- How many proteins/GO terms exist as graph nodes in `test_data` -- and whether that matches `train_data`'s node counts (it may not; splits can carry different-sized node tables even in a shared, transductive graph).
- How many of those proteins actually have >=1 ground-truth BP annotation in test (this is almost certainly the source of the "streaming 32,675 proteins" number every evaluation run has printed -- confirmed here rather than assumed).
- How many distinct GO terms actually appear as a true label in test, out of the 27,855 possible.
- Annotations-per-protein and proteins-per-GO-term distributions -- the standard class-imbalance numbers a dataset-description section needs.
- What other node types (Disease, Drug, HPO, etc.) are present in `test_data`, confirming the same auxiliary graph context is available at test time as during training.

In [ ]:
from src.data.graph_builder import build_annotation_matrix
import torch

print('=' * 70)
print('TEST SPLIT -- FULL DATA OVERVIEW (GO_term_P / BP)')
print('=' * 70)

# -- Node counts: test_data vs train_data, for every node type present --
print('\n-- Node counts (test_data) --')
for ntype in test_data.node_types:
    n_test = test_data[ntype].x.shape[0] if hasattr(test_data[ntype], 'x') and test_data[ntype].x is not None else None
    n_train = train_data[ntype].x.shape[0] if ntype in train_data.node_types and hasattr(train_data[ntype], 'x') and train_data[ntype].x is not None else None
    match = 'same as train' if n_test == n_train else f'DIFFERS from train ({n_train:,})' if n_train is not None else 'not in train_data'
    print(f'  {ntype}: {n_test:,} nodes   ({match})' if n_test is not None else f'  {ntype}: no x features')

# -- Positive (protein, GO) supervision edges actually in test --
row, col, n_p_declared, n_go_declared = build_annotation_matrix(test_data, 'GO_term_P')
print(f'\n-- Supervision edges (test_data, GO_term_P) --')
print(f'  Declared protein node count:  {n_p_declared:,}')
print(f'  Declared GO_term_P node count: {n_go_declared:,}')
print(f'  Positive (protein, GO) edges:  {len(row):,}')

unique_test_proteins = sorted(set(row.tolist()))
unique_test_go = sorted(set(col.tolist()))
print(f'\n  Unique proteins WITH >=1 test annotation: {len(unique_test_proteins):,}')
print(f'  (this should match the "streaming N proteins" figure evaluate_all prints)')
print(f'  Unique GO terms actually used as a true label in test: {len(unique_test_go):,} of {n_go_declared:,} possible')

# -- Annotations-per-protein distribution --
from collections import Counter
per_protein = Counter(row.tolist())
counts = torch.tensor(list(per_protein.values()), dtype=torch.float32)
print(f'\n-- Annotations per protein (test, proteins with >=1 annotation only) --')
print(f'  min={counts.min():.0f}  max={counts.max():.0f}  mean={counts.mean():.2f}  '
      f'median={counts.median():.0f}  std={counts.std():.2f}')

# -- Proteins-per-GO-term distribution (class imbalance) --
per_go = Counter(col.tolist())
go_counts = torch.tensor(list(per_go.values()), dtype=torch.float32)
print(f'\n-- Proteins per GO term (test, terms with >=1 protein only) --')
print(f'  min={go_counts.min():.0f}  max={go_counts.max():.0f}  mean={go_counts.mean():.2f}  '
      f'median={go_counts.median():.0f}  std={go_counts.std():.2f}')
print(f'  GO terms with exactly 1 test protein (long-tail/rare terms): '
      f'{sum(1 for v in per_go.values() if v == 1):,} of {len(per_go):,}')

# -- Edge types (relations) available in test_data, for the auxiliary-graph-context claim --
print(f'\n-- Relation types present in test_data ({len(test_data.edge_types)} total) --')
rel_names = sorted(set(r for _, r, _ in test_data.edge_types))
for r in rel_names:
    print(f'  {r}')

## OFAT weight search: adv → dm → anchor, then a tuned final run

Sequential one-factor-at-a-time search, 15-epoch trials per stage. Superseded in rigor by the Optuna joint search further down, but this produced the first working `{adv: 0.3, dm: 0.1, anchor: 1.5}` weights and the checkpoint (`generator_completion_tuned.pt`) everything after it builds on.

In [ ]:
raise RuntimeError(
    'This OFAT search already ran and its result is saved '
    '(generator_completion_tuned.pt on Drive) -- see the OFAT markdown section '
    'above for the recorded numbers. You almost certainly want the OPTUNA cell '
    'further down instead (look for "Trial N: adv=..." in its output, not '
    '"STAGE A"). If you genuinely mean to re-run this OFAT search from scratch, '
    'delete this raise statement first.'
)

import torch, os, copy, time, traceback
from src.models.compgcn import CompGCN
from src.models.generator import Generator
from src.models.discriminator import Discriminator
from src.training.adversarial import train_adversarial
from src.evaluation.metrics import evaluate_all

locked_ckpt = os.path.join(CHECKPOINT_DIR, 'compgcn_pretrained_full_graph_LOCKED.pt')
SEARCH_EPOCHS = 15   # short trials, ranking only
CHECK_GW = 0.85      # single gen_weight used to rank candidates during search

search_log = {}
t0 = time.time()

def elapsed():
    return f'{(time.time() - t0) / 60:.1f} min'

def run_trial(weights, epochs=SEARCH_EPOCHS):
    """Train a fresh completion-style generator with given loss weights, return Fmax at CHECK_GW."""
    enc = CompGCN(train_data, cfg).to(DEVICE)
    enc.load_state_dict(torch.load(locked_ckpt, map_location=DEVICE))
    gen = Generator(cfg).to(DEVICE)
    disc = Discriminator(cfg).to(DEVICE)

    trial_cfg = dict(cfg)
    trial_cfg['adversarial'] = dict(cfg['adversarial'])
    trial_cfg['adversarial']['epochs'] = epochs

    enc, gen, disc = train_adversarial(
        encoder=enc, generator=gen, discriminator=disc,
        distmult=distmult, train_data=train_data, val_data=val_data,
        ancestor_table=ancestor_table, cfg=trial_cfg, device=DEVICE,
        logger=None, checkpoint_dir=None, freeze_encoder=True,
        gen_loss_weights=weights,
    )

    eval_cfg = copy.deepcopy(cfg)
    eval_cfg['evaluation']['gen_weight'] = CHECK_GW
    r = evaluate_all(
        encoder=enc, generator=gen, distmult=distmult, data=val_data,
        ancestor_table=ancestor_table, target_type=target_type, cfg=eval_cfg,
        device=DEVICE, mode='encoder', ic_vec=ic_vecs.get(cfg['data']['ontology']),
    )
    return r['fmax']

# ── STAGE A: sweep adv, hold dm=0.1, anchor=1.0 ──
print('=' * 70); print(f'STAGE A: sweeping adv (dm=0.1, anchor=1.0 fixed), {SEARCH_EPOCHS} epochs each'); print('=' * 70)
adv_candidates = [0.0, 0.05, 0.1, 0.2, 0.3]
adv_results = {}
for adv in adv_candidates:
    try:
        fmax = run_trial({'adv': adv, 'dm': 0.1, 'anchor': 1.0})
        adv_results[adv] = fmax
        search_log[f'adv={adv}'] = fmax
        print(f'  adv={adv:.2f}: Fmax@gw={CHECK_GW}={fmax:.4f}   ({elapsed()})')
    except Exception:
        print(f'  adv={adv:.2f} FAILED:'); traceback.print_exc()

best_adv = max(adv_results, key=adv_results.get) if adv_results else 0.1
print(f'\nBest adv = {best_adv} (Fmax={adv_results.get(best_adv, float("nan")):.4f})\n')

# ── STAGE B: sweep dm, hold adv=best_adv, anchor=1.0 ──
print('=' * 70); print(f'STAGE B: sweeping dm (adv={best_adv}, anchor=1.0 fixed)'); print('=' * 70)
dm_candidates = [0.0, 0.05, 0.1, 0.2, 0.3]
dm_results = {}
for dm in dm_candidates:
    try:
        fmax = run_trial({'adv': best_adv, 'dm': dm, 'anchor': 1.0})
        dm_results[dm] = fmax
        search_log[f'dm={dm} (adv={best_adv})'] = fmax
        print(f'  dm={dm:.2f}: Fmax@gw={CHECK_GW}={fmax:.4f}   ({elapsed()})')
    except Exception:
        print(f'  dm={dm:.2f} FAILED:'); traceback.print_exc()

best_dm = max(dm_results, key=dm_results.get) if dm_results else 0.1
print(f'\nBest dm = {best_dm} (Fmax={dm_results.get(best_dm, float("nan")):.4f})\n')

# ── STAGE C: sweep anchor, hold adv=best_adv, dm=best_dm ──
print('=' * 70); print(f'STAGE C: sweeping anchor (adv={best_adv}, dm={best_dm} fixed)'); print('=' * 70)
anchor_candidates = [0.5, 1.0, 1.5, 2.0]
anchor_results = {}
for anchor in anchor_candidates:
    try:
        fmax = run_trial({'adv': best_adv, 'dm': best_dm, 'anchor': anchor})
        anchor_results[anchor] = fmax
        search_log[f'anchor={anchor} (adv={best_adv}, dm={best_dm})'] = fmax
        print(f'  anchor={anchor:.2f}: Fmax@gw={CHECK_GW}={fmax:.4f}   ({elapsed()})')
    except Exception:
        print(f'  anchor={anchor:.2f} FAILED:'); traceback.print_exc()

best_anchor = max(anchor_results, key=anchor_results.get) if anchor_results else 1.0
print(f'\nBest anchor = {best_anchor} (Fmax={anchor_results.get(best_anchor, float("nan")):.4f})\n')

best_weights = {'adv': best_adv, 'dm': best_dm, 'anchor': best_anchor}
print(f'WINNING WEIGHTS: {best_weights}')

# ── FINAL: full-length run with the winning combination + full sweep + test-set confirmation ──
print('=' * 70); print(f'FINAL RUN: {best_weights}, 40 epochs, full sweep + test-set confirmation'); print('=' * 70)
try:
    final_encoder = CompGCN(train_data, cfg).to(DEVICE)
    final_encoder.load_state_dict(torch.load(locked_ckpt, map_location=DEVICE))
    final_generator = Generator(cfg).to(DEVICE)
    final_discriminator = Discriminator(cfg).to(DEVICE)

    final_cfg = dict(cfg)
    final_cfg['adversarial'] = dict(cfg['adversarial'])
    final_cfg['adversarial']['epochs'] = 40

    final_encoder, final_generator, final_discriminator = train_adversarial(
        encoder=final_encoder, generator=final_generator, discriminator=final_discriminator,
        distmult=distmult, train_data=train_data, val_data=val_data,
        ancestor_table=ancestor_table, cfg=final_cfg, device=DEVICE,
        logger=None, checkpoint_dir=None, freeze_encoder=True,
        gen_loss_weights=best_weights,
    )

    final_ckpt_path = os.path.join(CHECKPOINT_DIR, 'generator_completion_tuned.pt')
    torch.save({
        'encoder': final_encoder.state_dict(), 'generator': final_generator.state_dict(),
        'discriminator': final_discriminator.state_dict(), 'weights': best_weights,
    }, final_ckpt_path)
    print(f'Saved -> {final_ckpt_path}')

    print('\n-- VAL sweep --')
    val_final = {}
    for gw in [0.0, 0.3, 0.5, 0.7, 0.85, 1.0]:
        sweep_cfg = copy.deepcopy(cfg)
        sweep_cfg['evaluation']['gen_weight'] = gw
        r = evaluate_all(
            encoder=final_encoder, generator=final_generator, distmult=distmult, data=val_data,
            ancestor_table=ancestor_table, target_type=target_type, cfg=sweep_cfg,
            device=DEVICE, mode='encoder', ic_vec=ic_vecs.get(cfg['data']['ontology']),
        )
        val_final[gw] = r
        print(f"  [VAL] gen_weight={gw:.2f}: Fmax={r['fmax']:.4f}  MCC={r['mcc']:.4f}  MicroF1={r['micro_f1']:.4f}")

    best_gw_final = max(val_final, key=lambda k: val_final[k]['fmax'])
    print(f"\nBest gen_weight on val: {best_gw_final} (Fmax={val_final[best_gw_final]['fmax']:.4f})")

    print('\n-- TEST-SET confirmation --')
    test_final = {}
    for gw in sorted(set([0.0, 0.85, 1.0, best_gw_final])):
        test_cfg = copy.deepcopy(cfg)
        test_cfg['evaluation']['gen_weight'] = gw
        r = evaluate_all(
            encoder=final_encoder, generator=final_generator, distmult=distmult, data=test_data,
            ancestor_table=ancestor_table, target_type=target_type, cfg=test_cfg,
            device=DEVICE, mode='encoder', ic_vec=ic_vecs.get(cfg['data']['ontology']),
        )
        test_final[gw] = r
        print(f"  [TEST] gen_weight={gw:.2f}: Fmax={r['fmax']:.4f}  MCC={r['mcc']:.4f}  MicroF1={r['micro_f1']:.4f}")
except Exception:
    print('FINAL RUN FAILED:')
    traceback.print_exc()

# ── SUMMARY ──
print('=' * 70); print('WEIGHT SEARCH COMPLETE -- SUMMARY'); print('=' * 70)
for k, v in search_log.items():
    print(f'  {k}: {v:.4f}')
print(f'\nWinning weights: {best_weights}')
print(f'Total time: {elapsed()}')

STAGE A: sweeping adv (dm=0.1, anchor=1.0 fixed), 15 epochs each
[rel_idx] available relations: ['Orthology', 'Pathway', 'kegg_path_prot', 'domain_function', 'function_function', 'protein_domain', 'PPI', 'HPO', 'kegg_dis_prot', 'Disease', 'Drug', 'kegg_dis_path', 'protein_ec', 'hpodis', 'kegg_dis_drug', 'Chembl', 'protein_function', 'rev_Orthology', 'rev_Pathway', 'rev_kegg_path_prot', 'rev_domain_function', 'rev_function_function', 'rev_protein_domain', 'rev_PPI', 'rev_HPO', 'rev_kegg_dis_prot', 'rev_Disease', 'rev_Drug', 'rev_kegg_dis_path', 'rev_protein_ec', 'rev_hpodis', 'rev_kegg_dis_drug', 'rev_Chembl', 'rev_protein_function']
[rel_idx] selected 'protein_function' → 16
[NegativeSampler] tiers — easy: 9,481  medium: 9,403  hard: 8,971
Starting adversarial training: 15 epochs, 797,081 positive pairs
  WGAN + spectral norm | n_critic=5 | beta1=0.0 beta2=0.9
[Adv 1/15] W_dist=0.314  C_loss=-0.314  G_loss=0.368  E_loss=0.000  anchor=0.306  E_anchor=0.000  scores(real=0.83 fake=0.74 ha

### Resume: Step 1 already confirmed, skip straight to ablation + REINFORCE

The cell above's Step 1 (test-set confirmation) completed and printed the official result before the runtime disconnected during Step 2 (no-adversarial ablation, interrupted at epoch 17/40, nothing checkpointed yet -- no progress lost, just wasted compute time). This cell hardcodes Step 1's already-confirmed numbers and resumes directly at Step 2, then runs Step 3, so we don't spend another ~hour recomputing test-set evals we already have.

In [ ]:
import torch, os, copy, time, traceback
from src.models.compgcn import CompGCN
from src.models.generator import Generator
from src.models.discriminator import Discriminator
from src.models.reward import RewardModule
from src.training.adversarial import train_adversarial
from src.training.rl_trainer import train_rl_reinforce
from src.evaluation.metrics import evaluate_all

final_log = {}
t_start = time.time()
locked_ckpt = os.path.join(CHECKPOINT_DIR, 'compgcn_pretrained_full_graph_LOCKED.pt')
tuned_ckpt_path = os.path.join(CHECKPOINT_DIR, 'generator_completion_tuned.pt')

def elapsed():
    return f'{(time.time() - t_start) / 60:.1f} min'

# ── STEP 1 (already confirmed in the earlier run, hardcoded here so it isn't recomputed) ──
print('=' * 70); print('STEP 1: TEST-SET CONFIRMATION (already confirmed -- hardcoded)'); print('=' * 70)
test_final = {
    0.00: {'fmax': 0.5501, 'smin': 203.2621, 'aupr': 0.3890, 'auroc': 0.9765, 'micro_f1': 0.3756, 'macro_f1': 0.0420, 'mcc': 0.3894},
    0.85: {'fmax': 0.7085, 'smin': 161.9997, 'aupr': 0.6385, 'auroc': 0.9899, 'micro_f1': 0.5447, 'macro_f1': 0.1470, 'mcc': 0.5500},
    1.00: {'fmax': 0.7207, 'smin': 172.5642, 'aupr': 0.5916, 'auroc': 0.9883, 'micro_f1': 0.4292, 'macro_f1': 0.1439, 'mcc': 0.4655},
}
for gw, r in test_final.items():
    final_log[f'TEST tuned completion gen_weight={gw:.2f}'] = r
    print(f"  [TEST] gen_weight={gw:.2f}: Fmax={r['fmax']:.4f}  MCC={r['mcc']:.4f}  MicroF1={r['micro_f1']:.4f}")
print()
print("OFFICIAL RESULT -- weights={'adv': 0.3, 'dm': 0.1, 'anchor': 1.5}, gen_weight=0.85")
off = test_final[0.85]
print(f"  Fmax={off['fmax']:.4f}  Smin={off['smin']:.4f}  AUPR={off['aupr']:.4f}  AUROC={off['auroc']:.4f}  "
      f"Micro-F1={off['micro_f1']:.4f}  Macro-F1={off['macro_f1']:.4f}  MCC={off['mcc']:.4f}")
print()

# ── STEP 2: No-adversarial ablation at the TUNED base (adv=0.0, dm=0.1, anchor=1.5) ──
try:
    print('=' * 70); print('STEP 2: NO-ADVERSARIAL ABLATION (tuned base)'); print('=' * 70)
    noadv_encoder = CompGCN(train_data, cfg).to(DEVICE)
    noadv_encoder.load_state_dict(torch.load(locked_ckpt, map_location=DEVICE))
    noadv_generator = Generator(cfg).to(DEVICE)
    noadv_discriminator = Discriminator(cfg).to(DEVICE)

    noadv_cfg = dict(cfg)
    noadv_cfg['adversarial'] = dict(cfg['adversarial'])
    noadv_cfg['adversarial']['epochs'] = 40

    noadv_encoder, noadv_generator, noadv_discriminator = train_adversarial(
        encoder=noadv_encoder, generator=noadv_generator, discriminator=noadv_discriminator,
        distmult=distmult, train_data=train_data, val_data=val_data,
        ancestor_table=ancestor_table, cfg=noadv_cfg, device=DEVICE,
        logger=None, checkpoint_dir=None, freeze_encoder=True,
        gen_loss_weights={'adv': 0.0, 'dm': 0.1, 'anchor': 1.5},
    )

    noadv_ckpt_path = os.path.join(CHECKPOINT_DIR, 'generator_completion_noadv_tuned.pt')
    torch.save({
        'encoder': noadv_encoder.state_dict(), 'generator': noadv_generator.state_dict(),
        'discriminator': noadv_discriminator.state_dict(),
    }, noadv_ckpt_path)
    print(f'Saved -> {noadv_ckpt_path}')

    for gw in [0.0, 0.3, 0.5, 0.7, 0.85, 1.0]:
        sweep_cfg = copy.deepcopy(cfg)
        sweep_cfg['evaluation']['gen_weight'] = gw
        r = evaluate_all(
            encoder=noadv_encoder, generator=noadv_generator, distmult=distmult, data=val_data,
            ancestor_table=ancestor_table, target_type=target_type, cfg=sweep_cfg,
            device=DEVICE, mode='encoder', ic_vec=ic_vecs.get(cfg['data']['ontology']),
        )
        final_log[f'VAL no-adv(tuned) gen_weight={gw:.2f}'] = r
        print(f"  [VAL, no-adv/tuned] gen_weight={gw:.2f}: Fmax={r['fmax']:.4f}  MCC={r['mcc']:.4f}  MicroF1={r['micro_f1']:.4f}")
    print(f'Step 2 done at {elapsed()}')
except Exception:
    print('STEP 2 FAILED:')
    traceback.print_exc()
print()

# ── STEP 3: One more REINFORCE attempt, warm-started from the TUNED completion Generator ──
try:
    print('=' * 70); print('STEP 3: REINFORCE PHASE 3 (warm-started from tuned completion)'); print('=' * 70)
    tuned_ckpt = torch.load(tuned_ckpt_path, map_location=DEVICE)
    rl_encoder = CompGCN(train_data, cfg).to(DEVICE)
    rl_encoder.load_state_dict(torch.load(locked_ckpt, map_location=DEVICE))
    rl_generator = Generator(cfg).to(DEVICE)
    rl_generator.load_state_dict(tuned_ckpt['generator'])
    rl_discriminator = Discriminator(cfg).to(DEVICE)
    rl_discriminator.load_state_dict(tuned_ckpt['discriminator'])
    print('Warm-started RL Generator/Critic from generator_completion_tuned.pt')

    reward_module = RewardModule(cfg, distmult, rl_discriminator).to(DEVICE)

    rl_cfg = dict(cfg)
    rl_cfg['rl'] = dict(cfg['rl'])
    rl_cfg['rl']['epochs'] = 30
    rl_cfg['rl']['policy_sigma'] = 0.1

    rl_encoder, rl_generator = train_rl_reinforce(
        encoder=rl_encoder, generator=rl_generator, distmult=distmult,
        reward_module=reward_module, train_data=train_data, val_data=val_data,
        ancestor_table=ancestor_table, cfg=rl_cfg, device=DEVICE,
        checkpoint_dir=None, logger=None,
    )

    rl_ckpt_path = os.path.join(CHECKPOINT_DIR, 'rl_reinforce_tuned.pt')
    torch.save({'encoder': rl_encoder.state_dict(), 'generator': rl_generator.state_dict()}, rl_ckpt_path)
    print(f'Saved -> {rl_ckpt_path}')

    for gw in [0.0, 0.5, 0.85, 1.0]:
        sweep_cfg = copy.deepcopy(cfg)
        sweep_cfg['evaluation']['gen_weight'] = gw
        r = evaluate_all(
            encoder=rl_encoder, generator=rl_generator, distmult=distmult, data=val_data,
            ancestor_table=ancestor_table, target_type=target_type, cfg=sweep_cfg,
            device=DEVICE, mode='encoder', ic_vec=ic_vecs.get(cfg['data']['ontology']),
        )
        final_log[f'VAL reinforce(tuned) gen_weight={gw:.2f}'] = r
        print(f"  [VAL, REINFORCE/tuned] gen_weight={gw:.2f}: Fmax={r['fmax']:.4f}  MCC={r['mcc']:.4f}  MicroF1={r['micro_f1']:.4f}")
    print(f'Step 3 done at {elapsed()}')
except Exception:
    print('STEP 3 FAILED:')
    traceback.print_exc()
print()

# ── FINAL SUMMARY ──
print('=' * 70); print('FINAL PASS COMPLETE -- SUMMARY'); print('=' * 70)
for k, v in final_log.items():
    print(f"  {k}: Fmax={v['fmax']:.4f}  MCC={v['mcc']:.4f}")
print(f'\nTotal time: {elapsed()}')

### Optuna: joint search over loss weights + training hyperparameters

The OFAT search only ever explored a narrow band around the original guess, one weight at a time (`adv`/`dm`/`anchor`), and it never touched anything about *how* the Generator/Critic train -- learning rates, `n_critic`, negative-sampling temperature. This cell replaces that with a single Optuna study (TPE sampler) over all seven parameters jointly, with wider ranges than OFAT used.

**Important scoping note:** the training loop's internal `val_Fmax` monitor (printed every 5 epochs during training, e.g. `val_Fmax=0.4157`) only scores the frozen *encoder* via ComplEx -- it never looks at the Generator at all. Since the encoder is identical and frozen across every trial here, that internal metric would be flat regardless of the Generator's hyperparameters, so it CANNOT be used for early-stopping/pruning without measuring the wrong thing. This search deliberately skips pruning and scores each trial only via the real `evaluate_all(mode='encoder', gen_weight=0.85)` at the end of a fixed 15-epoch trial -- the same call and trial length OFAT used, so trial cost is directly comparable (~80 min/trial observed previously).

**Objective is Fmax+MCC, not Fmax alone** -- the OFAT search optimized only val Fmax and that let val MCC regress at the chosen operating point (a regression that didn't hold on test, but was still a real blind spot in the search itself). The Optuna objective here is `0.5*Fmax + 0.5*MCC`, so a candidate can't win by trading one for the other.

**Resumable across disconnects** -- the study is backed by a SQLite file on Drive (`optuna_completion_search.db`), so completed trials are never lost. If the runtime disconnects mid-search, just reconnect, re-run FULL RESET RECOVERY, `pip install optuna`, and re-run this cell -- `load_if_exists=True` picks up exactly where it left off (only the trial that was actually in-flight at the moment of disconnect is wasted, nothing already completed is).

Search space:

| Parameter | Range | OFAT covered |
|---|---|---|
| `adv` (loss weight) | 0.0 - 1.0 | 0.0 - 0.3 |
| `dm` (loss weight) | 0.0 - 0.5 | 0.0 - 0.3 |
| `anchor` (loss weight) | 0.1 - 3.0 | 0.5 - 2.0 |
| `lr_generator` | 1e-5 - 1e-3 (log) | fixed at 1e-4 |
| `lr_critic` | 1e-5 - 1e-3 (log) | fixed at 1e-4 |
| `n_critic` | 1 - 5 (int) | fixed at 5 |
| `hard_neg_temperature` | 0.1 - 1.0 | fixed at 0.5 |

**Cut down from the original plan (2026-09-20) to fit the time available:** `n_critic`'s range was capped at 1-5 instead of 1-10 (5 was already the known-good default; higher values were unvalidated territory, not a proven improvement, and were the single biggest driver of worst-case trial cost), and `N_TRIALS_TOTAL` was lowered from 40 to 25. Estimated total: roughly **25-30 hours**, still spanning a few Colab sessions, down from the original ~55-60 hour estimate. Note this reduction in the paper's methodology section as a deliberate time-budget trade-off, not a limitation discovered after the fact. Re-run this same cell after each reconnect to continue -- already-completed trials are never lost.

In [ ]:
!pip install optuna -q

import optuna, torch, os, copy, time, traceback
from src.models.compgcn import CompGCN
from src.models.generator import Generator
from src.models.discriminator import Discriminator
from src.training.adversarial import train_adversarial
from src.evaluation.metrics import evaluate_all

locked_ckpt = os.path.join(CHECKPOINT_DIR, 'compgcn_pretrained_full_graph_LOCKED.pt')
STUDY_DB = os.path.join(CHECKPOINT_DIR, 'optuna_completion_search.db')
TRIAL_EPOCHS = 15   # matches OFAT's trial length -- keeps per-trial cost comparable/known
CHECK_GW = 0.85     # the chosen operating point, held fixed during this search
N_TRIALS_TOTAL = 25 # raise or lower any time; already-completed trials are never lost

t0 = time.time()
def elapsed():
    return f'{(time.time() - t0) / 60:.1f} min'

def objective(trial):
    weights = {
        'adv': trial.suggest_float('adv', 0.0, 1.0),
        'dm': trial.suggest_float('dm', 0.0, 0.5),
        'anchor': trial.suggest_float('anchor', 0.1, 3.0),
    }
    lr_generator = trial.suggest_float('lr_generator', 1e-5, 1e-3, log=True)
    lr_critic = trial.suggest_float('lr_critic', 1e-5, 1e-3, log=True)
    n_critic = trial.suggest_int('n_critic', 1, 5)
    hard_neg_temperature = trial.suggest_float('hard_neg_temperature', 0.1, 1.0)

    enc = CompGCN(train_data, cfg).to(DEVICE)
    enc.load_state_dict(torch.load(locked_ckpt, map_location=DEVICE))
    gen = Generator(cfg).to(DEVICE)
    disc = Discriminator(cfg).to(DEVICE)

    trial_cfg = dict(cfg)
    trial_cfg['adversarial'] = dict(cfg['adversarial'])
    trial_cfg['adversarial']['epochs'] = TRIAL_EPOCHS
    trial_cfg['adversarial']['lr_generator'] = lr_generator
    trial_cfg['adversarial']['lr_critic'] = lr_critic
    trial_cfg['adversarial']['n_critic'] = n_critic
    trial_cfg['adversarial']['negative_sampling'] = {'hard_neg_temperature': hard_neg_temperature}

    try:
        enc, gen, disc = train_adversarial(
            encoder=enc, generator=gen, discriminator=disc,
            distmult=distmult, train_data=train_data, val_data=val_data,
            ancestor_table=ancestor_table, cfg=trial_cfg, device=DEVICE,
            logger=None, checkpoint_dir=None, freeze_encoder=True,
            gen_loss_weights=weights,
        )
    except Exception:
        print(f'  Trial {trial.number} training FAILED:')
        traceback.print_exc()
        raise optuna.TrialPruned()

    eval_cfg = copy.deepcopy(cfg)
    eval_cfg['evaluation']['gen_weight'] = CHECK_GW
    r = evaluate_all(
        encoder=enc, generator=gen, distmult=distmult, data=val_data,
        ancestor_table=ancestor_table, target_type=target_type, cfg=eval_cfg,
        device=DEVICE, mode='encoder', ic_vec=ic_vecs.get(cfg['data']['ontology']),
    )
    score = 0.5 * r['fmax'] + 0.5 * r['mcc']
    trial.set_user_attr('fmax', r['fmax'])
    trial.set_user_attr('mcc', r['mcc'])
    trial.set_user_attr('micro_f1', r['micro_f1'])
    print(f"  Trial {trial.number}: adv={weights['adv']:.3f} dm={weights['dm']:.3f} "
          f"anchor={weights['anchor']:.3f} lr_g={lr_generator:.2e} lr_c={lr_critic:.2e} "
          f"n_critic={n_critic} temp={hard_neg_temperature:.2f} "
          f"-> Fmax={r['fmax']:.4f} MCC={r['mcc']:.4f} score={score:.4f}  ({elapsed()})")
    return score

study = optuna.create_study(
    study_name='completion_weights_v1',
    storage=f'sqlite:///{STUDY_DB}',
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
    load_if_exists=True,
)
print(f'Study loaded from {STUDY_DB}')
print(f'{len(study.trials)} trial(s) already completed.')

remaining = max(0, N_TRIALS_TOTAL - len(study.trials))
print(f'Running {remaining} more trial(s) (target {N_TRIALS_TOTAL} total)...\n')
study.optimize(objective, n_trials=remaining)

print('\n' + '=' * 70)
print('OPTUNA SEARCH -- BEST TRIAL SO FAR')
print('=' * 70)
print(f'  Score (0.5*Fmax + 0.5*MCC): {study.best_value:.4f}')
print(f'  Params: {study.best_params}')
print(f"  Fmax: {study.best_trial.user_attrs.get('fmax'):.4f}  "
      f"MCC: {study.best_trial.user_attrs.get('mcc'):.4f}  "
      f"MicroF1: {study.best_trial.user_attrs.get('micro_f1'):.4f}")
print(f'\nTotal trials in study: {len(study.trials)}')
print(f'Total time this session: {elapsed()}')

### Run this once you're satisfied with the search

The cell above only *searches* -- each trial is a cheap 15-epoch proxy, and it never retrains a full model. This cell takes whatever `study.best_params` currently is (you don't have to wait for `N_TRIALS_TOTAL` -- run this any time you want to check where the search stands), retrains a full 40-epoch model with those weights AND hyperparameters, then runs the same VAL sweep + TEST-set confirmation as the OFAT search did -- so the result is directly comparable to the OFAT tuned run (test Fmax=0.7085, MCC=0.5500 at gen_weight=0.85).

Picks the operating `gen_weight` by the same composite score the search itself optimizes (`0.5*Fmax + 0.5*MCC`), not by Fmax alone -- consistent with wanting a config that does well overall rather than one number.

Safe to re-run later with more completed trials -- it always reads `study.best_params` fresh from the SQLite file, so a better result found since the last time you ran this cell will be picked up automatically.

In [ ]:
import optuna, torch, os, copy
from src.models.compgcn import CompGCN
from src.models.generator import Generator
from src.models.discriminator import Discriminator
from src.training.adversarial import train_adversarial
from src.evaluation.metrics import evaluate_all

locked_ckpt = os.path.join(CHECKPOINT_DIR, 'compgcn_pretrained_full_graph_LOCKED.pt')
STUDY_DB = os.path.join(CHECKPOINT_DIR, 'optuna_completion_search.db')

study = optuna.load_study(study_name='completion_weights_v1', storage=f'sqlite:///{STUDY_DB}')
print(f'Loaded study: {len(study.trials)} completed trial(s).')
best = study.best_params
print(f'Best params so far: {best}')
print(f"Best trial -- Fmax={study.best_trial.user_attrs.get('fmax'):.4f}  "
      f"MCC={study.best_trial.user_attrs.get('mcc'):.4f}  score={study.best_value:.4f}")

best_weights = {'adv': best['adv'], 'dm': best['dm'], 'anchor': best['anchor']}

final_encoder = CompGCN(train_data, cfg).to(DEVICE)
final_encoder.load_state_dict(torch.load(locked_ckpt, map_location=DEVICE))
final_generator = Generator(cfg).to(DEVICE)
final_discriminator = Discriminator(cfg).to(DEVICE)

final_cfg = dict(cfg)
final_cfg['adversarial'] = dict(cfg['adversarial'])
final_cfg['adversarial']['epochs'] = 40
final_cfg['adversarial']['lr_generator'] = best['lr_generator']
final_cfg['adversarial']['lr_critic'] = best['lr_critic']
final_cfg['adversarial']['n_critic'] = best['n_critic']
final_cfg['adversarial']['negative_sampling'] = {'hard_neg_temperature': best['hard_neg_temperature']}

print('\n' + '=' * 70)
print(f'FINAL RUN: {best_weights} + tuned hyperparams, 40 epochs')
print('=' * 70)

final_encoder, final_generator, final_discriminator = train_adversarial(
    encoder=final_encoder, generator=final_generator, discriminator=final_discriminator,
    distmult=distmult, train_data=train_data, val_data=val_data,
    ancestor_table=ancestor_table, cfg=final_cfg, device=DEVICE,
    logger=None, checkpoint_dir=None, freeze_encoder=True,
    gen_loss_weights=best_weights,
)

final_ckpt_path = os.path.join(CHECKPOINT_DIR, 'generator_completion_optuna.pt')
torch.save({
    'encoder': final_encoder.state_dict(), 'generator': final_generator.state_dict(),
    'discriminator': final_discriminator.state_dict(),
    'weights': best_weights, 'optuna_params': best,
}, final_ckpt_path)
print(f'Saved -> {final_ckpt_path}')

print('\n-- VAL sweep --')
val_final = {}
for gw in [0.0, 0.3, 0.5, 0.7, 0.85, 1.0]:
    sweep_cfg = copy.deepcopy(cfg)
    sweep_cfg['evaluation']['gen_weight'] = gw
    r = evaluate_all(
        encoder=final_encoder, generator=final_generator, distmult=distmult, data=val_data,
        ancestor_table=ancestor_table, target_type=target_type, cfg=sweep_cfg,
        device=DEVICE, mode='encoder', ic_vec=ic_vecs.get(cfg['data']['ontology']),
    )
    val_final[gw] = r
    print(f"  [VAL] gen_weight={gw:.2f}: Fmax={r['fmax']:.4f}  MCC={r['mcc']:.4f}  MicroF1={r['micro_f1']:.4f}")

best_gw_final = max(val_final, key=lambda k: 0.5 * val_final[k]['fmax'] + 0.5 * val_final[k]['mcc'])
print(f'\nBest gen_weight on val (by 0.5*Fmax + 0.5*MCC): {best_gw_final}')

print('\n-- TEST-SET confirmation --')
test_final = {}
for gw in sorted(set([0.0, 0.85, 1.0, best_gw_final])):
    test_cfg = copy.deepcopy(cfg)
    test_cfg['evaluation']['gen_weight'] = gw
    r = evaluate_all(
        encoder=final_encoder, generator=final_generator, distmult=distmult, data=test_data,
        ancestor_table=ancestor_table, target_type=target_type, cfg=test_cfg,
        device=DEVICE, mode='encoder', ic_vec=ic_vecs.get(cfg['data']['ontology']),
    )
    test_final[gw] = r
    print(f"  [TEST] gen_weight={gw:.2f}: Fmax={r['fmax']:.4f}  MCC={r['mcc']:.4f}  "
          f"MicroF1={r['micro_f1']:.4f}  Smin={r['smin']:.4f}")

print('\nCompare against the OFAT tuned run: test Fmax=0.7085, MCC=0.5500, MicroF1=0.5447 at gen_weight=0.85.')

### Spot-check: does n_critic above 5 ever help?

The main search capped `n_critic` at 1-5 for time-budget reasons (see the search cell's note) -- 6-10 was never validated as an improvement anywhere in this project and was the single biggest driver of worst-case trial cost, so it was excluded from the random/TPE sampling entirely rather than left in to destabilize a 25-trial budget.

This is a **controlled follow-up, not a re-opening of the search**: it takes whatever `study.best_params` found (adv/dm/anchor/lr_generator/lr_critic/hard_neg_temperature all held fixed) and tries exactly three specific values -- `n_critic = 6, 8, 10` -- as single 15-epoch trials scored the same way the main search scores everything (`0.5*Fmax + 0.5*MCC` at gen_weight=0.85). That isolates the one question actually being asked (does raising n_critic help, holding everything else at its already-found-best) instead of letting it get confounded with randomly-varying other parameters.

**Read the result with the right amount of confidence:** this is one run per value, not a repeated ablation -- given the ~0.004-0.006 run-to-run noise floor already established in this project, it can rule out a *large* effect but can't reliably separate a small real effect from noise. Treat it as a spot-check that answers "is there an obvious win being missed," not as a definitive verdict on n_critic. It's also conditional -- it only tests whether higher n_critic helps *given* the other parameters already optimized for n_critic<=5, not whether some different combination would pair better with a higher n_critic. Both caveats are worth stating plainly if this goes in the paper, the same way OFAT's own sequential-search limitation already is.

Run this any time after the main search has produced a `study.best_params` -- doesn't need to wait for the full 25 trials.

In [ ]:
import optuna, torch, os, copy, time
from src.models.compgcn import CompGCN
from src.models.generator import Generator
from src.models.discriminator import Discriminator
from src.training.adversarial import train_adversarial
from src.evaluation.metrics import evaluate_all

locked_ckpt = os.path.join(CHECKPOINT_DIR, 'compgcn_pretrained_full_graph_LOCKED.pt')
STUDY_DB = os.path.join(CHECKPOINT_DIR, 'optuna_completion_search.db')
SPOTCHECK_EPOCHS = 15   # matches the main search's trial length -- directly comparable
SPOTCHECK_GW = 0.85
N_CRITIC_SPOTCHECK_VALUES = [6, 8, 10]

study = optuna.load_study(study_name='completion_weights_v1', storage=f'sqlite:///{STUDY_DB}')
best = study.best_params
print(f'Main search best (n_critic={best["n_critic"]}, capped at <=5): score={study.best_value:.4f}')
print(f'  Fmax={study.best_trial.user_attrs.get("fmax"):.4f}  MCC={study.best_trial.user_attrs.get("mcc"):.4f}')
print(f'Holding adv/dm/anchor/lr_generator/lr_critic/hard_neg_temperature fixed at these values,')
print(f'spot-checking n_critic in {N_CRITIC_SPOTCHECK_VALUES}...\n')

t0 = time.time()
def elapsed():
    return f'{(time.time() - t0) / 60:.1f} min'

spotcheck_results = {best['n_critic']: study.best_value}  # the main search's own winner, for the table

for nc in N_CRITIC_SPOTCHECK_VALUES:
    weights = {'adv': best['adv'], 'dm': best['dm'], 'anchor': best['anchor']}

    enc = CompGCN(train_data, cfg).to(DEVICE)
    enc.load_state_dict(torch.load(locked_ckpt, map_location=DEVICE))
    gen = Generator(cfg).to(DEVICE)
    disc = Discriminator(cfg).to(DEVICE)

    trial_cfg = dict(cfg)
    trial_cfg['adversarial'] = dict(cfg['adversarial'])
    trial_cfg['adversarial']['epochs'] = SPOTCHECK_EPOCHS
    trial_cfg['adversarial']['lr_generator'] = best['lr_generator']
    trial_cfg['adversarial']['lr_critic'] = best['lr_critic']
    trial_cfg['adversarial']['n_critic'] = nc
    trial_cfg['adversarial']['negative_sampling'] = {'hard_neg_temperature': best['hard_neg_temperature']}

    enc, gen, disc = train_adversarial(
        encoder=enc, generator=gen, discriminator=disc,
        distmult=distmult, train_data=train_data, val_data=val_data,
        ancestor_table=ancestor_table, cfg=trial_cfg, device=DEVICE,
        logger=None, checkpoint_dir=None, freeze_encoder=True,
        gen_loss_weights=weights,
    )

    eval_cfg = copy.deepcopy(cfg)
    eval_cfg['evaluation']['gen_weight'] = SPOTCHECK_GW
    r = evaluate_all(
        encoder=enc, generator=gen, distmult=distmult, data=val_data,
        ancestor_table=ancestor_table, target_type=target_type, cfg=eval_cfg,
        device=DEVICE, mode='encoder', ic_vec=ic_vecs.get(cfg['data']['ontology']),
    )
    score = 0.5 * r['fmax'] + 0.5 * r['mcc']
    spotcheck_results[nc] = score
    print(f'  n_critic={nc}: Fmax={r["fmax"]:.4f}  MCC={r["mcc"]:.4f}  score={score:.4f}   ({elapsed()})')

print('\n' + '=' * 70)
print('N_CRITIC SPOT-CHECK -- SUMMARY')
print('=' * 70)
for nc in sorted(spotcheck_results):
    tag = '  <- main search winner (capped range)' if nc == best['n_critic'] else ''
    print(f'  n_critic={nc:2d}: score={spotcheck_results[nc]:.4f}{tag}')

best_nc = max(spotcheck_results, key=spotcheck_results.get)
gain = spotcheck_results[best_nc] - study.best_value
if best_nc == best['n_critic']:
    print(f'\nNo higher n_critic beat the capped search\'s own winner -- the range cap did not cost anything here.')
elif gain < 0.005:
    print(f'\nn_critic={best_nc} scored marginally higher (+{gain:.4f}), but that\'s inside the ~0.004-0.006')
    print('run-to-run noise floor -- not distinguishable from noise on a single run each.')
else:
    print(f'\nn_critic={best_nc} beat the capped winner by {gain:.4f} -- larger than the noise floor,')
    print('worth a repeat run to confirm before concluding the cap cost real performance.')